Este codigo realiza un pegado de los datos para cada año con su respectivo modulo en un solo archivo por año

# 2007

In [13]:
import os, pandas as pd, unicodedata, zipfile, pyreadstat, tempfile, numpy as np

# =====================================================================
# CONFIGURACIÓN 2007 
# =====================================================================
ruta_base = r"E:\ENTREGA TESIS\Doctorado\Data\2007" 
carpeta_salida = r"E:\ENTREGA TESIS\Doctorado\Data\2007_" 

if not os.path.exists(carpeta_salida): os.makedirs(carpeta_salida)

meses = {'Enero':1,'Febrero':2,'Marzo':3,'Abril':4,'Mayo':5,'Junio':6,'Julio':7,'Agosto':8,'Septiembre':9,'Octubre':10,'Noviembre':11,'Diciembre':12}
zonas = {'Cabecera': (2, 'cabecera'), 'Resto': (3, 'resto')}
modulos_claves = {"caracteristicas_generales": "caract", "desocupados": "desocupados", "fuerza_trabajo": "fuerza", "inactivos": "inactivos", "ocupados": "ocupados", "otras_actividades": "otras", "otros_ingresos": "otros ingresos", "hogares": "vivienda"}

def quitar_tildes(cadena):
    s = ''.join(c for c in unicodedata.normalize('NFD', str(cadena)) if unicodedata.category(c) != 'Mn')
    return s.lower()

def encontrar_archivo_en_zip(lista_archivos_zip, zona_busqueda, palabra_clave):
    clave_norm = quitar_tildes(palabra_clave)
    for archivo in lista_archivos_zip:
        archivo_norm = quitar_tildes(str(archivo).lower())
        if '.dta' in archivo_norm and zona_busqueda in archivo_norm and clave_norm in archivo_norm:
            if clave_norm == "ocupados" and ("desocupado" in archivo_norm or "deocupado" in archivo_norm):
                continue
            return archivo
    return None

for nombre_modulo_final, palabra_clave in modulos_claves.items():
    print(f"\nProcesando 2007: {nombre_modulo_final.upper()}")
    datos_modulo = []
    for mes_nombre, mes_num in meses.items():
        ruta_zip = next((os.path.join(ruta_base, f) for f in [f"{mes_nombre}.dta.zip", f"{mes_nombre}.zip"] if os.path.exists(os.path.join(ruta_base, f))), None)
        
        if ruta_zip:
            try:
                with zipfile.ZipFile(ruta_zip, 'r') as z:
                    for zona_nombre, (zona_num, zona_busqueda) in zonas.items():
                        archivo_objetivo = encontrar_archivo_en_zip(z.namelist(), zona_busqueda, palabra_clave)
                        if archivo_objetivo:
                            tmp_path = ""
                            # 1. Extraer el archivo y CERRAR el bloque temporal
                            with tempfile.NamedTemporaryFile(delete=False, suffix=".dta") as tmp:
                                tmp.write(z.read(archivo_objetivo))
                                tmp_path = tmp.name
                            
                            # 2. Leer ahora que el archivo está cerrado (evita ReadstatError)
                            try:
                                df, meta = pyreadstat.read_dta(tmp_path, encoding="latin1") # latin1 es más seguro para 2007
                                df.columns = df.columns.str.lower()
                                df['mes'] = mes_num
                                df['zona_area'] = zona_num
                                
                                if nombre_modulo_final == "ocupados":
                                    intrusas = ['oficio1','ofico2','p7530','p760','rama2d_d','rama4d_d']
                                    df = df.drop(columns=[c for c in intrusas if c in df.columns])
                                
                                datos_modulo.append(df)
                                print(f"  [OK] {mes_nombre} - {zona_nombre}")
                            except Exception as e:
                                print(f"  [ERROR] Falló lectura de {archivo_objetivo} en {mes_nombre}: {e}")
                            finally:
                                if os.path.exists(tmp_path): os.remove(tmp_path)
            except Exception as e:
                print(f"  [AVISO] No se pudo abrir el zip {mes_nombre}: {e}")

    if datos_modulo:
        print(f"  -> Consolidando {nombre_modulo_final}...")
        df_final = pd.concat(datos_modulo, ignore_index=True)
        for col in df_final.columns:
            if df_final[col].dtype == 'object':
                df_final[col] = df_final[col].fillna('').astype(str)
        
        ruta_out = os.path.join(carpeta_salida, f"{nombre_modulo_final}.dta")
        df_final.to_stata(ruta_out, write_index=False, version=118)
        print(f"--> ¡GUARDADO EXITOSO! {len(df_final):,} registros")


Procesando 2007: CARACTERISTICAS_GENERALES
  [OK] Enero - Cabecera
  [OK] Enero - Resto
  [OK] Febrero - Cabecera
  [OK] Febrero - Resto
  [OK] Marzo - Cabecera
  [OK] Marzo - Resto
  [OK] Abril - Cabecera
  [OK] Abril - Resto
  [OK] Mayo - Cabecera
  [OK] Mayo - Resto
  [OK] Junio - Cabecera
  [OK] Junio - Resto
  [OK] Julio - Cabecera
  [OK] Julio - Resto
  [OK] Agosto - Cabecera
  [OK] Agosto - Resto
  [OK] Septiembre - Cabecera
  [OK] Septiembre - Resto
  [OK] Octubre - Cabecera
  [OK] Octubre - Resto
  [OK] Noviembre - Cabecera
  [OK] Noviembre - Resto
  [OK] Diciembre - Cabecera
  [OK] Diciembre - Resto
  -> Consolidando caracteristicas_generales...
--> ¡GUARDADO EXITOSO! 838,421 registros

Procesando 2007: DESOCUPADOS
  [OK] Enero - Cabecera
  [OK] Enero - Resto
  [OK] Febrero - Cabecera
  [OK] Febrero - Resto
  [OK] Marzo - Cabecera
  [OK] Marzo - Resto
  [OK] Abril - Cabecera
  [OK] Abril - Resto
  [OK] Mayo - Cabecera
  [OK] Mayo - Resto
  [OK] Junio - Cabecera
  [OK] Junio 

# 2008

In [1]:
import os
import pandas as pd
import unicodedata
import zipfile
import tempfile
import pyreadstat

# =====================================================================
# 1. CONFIGURACIÓN DE RUTAS
# =====================================================================
ruta_base = r"E:\ENTREGA TESIS\Doctorado\Data\2008" 
carpeta_salida = r"E:\ENTREGA TESIS\Doctorado\Data\2008_" 

if not os.path.exists(carpeta_salida):
    os.makedirs(carpeta_salida)
    print(f"Carpeta creada exitosamente en: {carpeta_salida}")

meses = {
    'Enero': 1, 'Febrero': 2, 'Marzo': 3, 'Abril': 4, 
    'Mayo': 5, 'Junio': 6, 'Julio': 7, 'Agosto': 8, 
    'Septiembre': 9, 'Octubre': 10, 'Noviembre': 11, 'Diciembre': 12
}

# OPCIÓN A: Solo Cabecera y Resto para evitar doble conteo de registros urbanos
zonas = {
    'Cabecera': (2, 'cabecera'),
    'Resto': (3, 'resto')
}

modulos_claves = {
    "caracteristicas_generales": "caract",
    "desocupados": "desocupados",
    "fuerza_trabajo": "fuerza",
    "inactivos": "inactivos",
    "ocupados": "ocupados", 
    "otras_actividades": "otras",
    "otros_ingresos": "otros ingresos",
    "hogares": "vivienda"
}

# =====================================================================
# 2. FUNCIONES DE APOYO
# =====================================================================
def quitar_tildes(cadena):
    s = ''.join(c for c in unicodedata.normalize('NFD', str(cadena)) if unicodedata.category(c) != 'Mn')
    return s.lower()

def encontrar_archivo_en_zip(lista_archivos_zip, zona_busqueda, palabra_clave):
    clave_norm = quitar_tildes(palabra_clave)
    for archivo in lista_archivos_zip:
        archivo_norm = quitar_tildes(str(archivo).lower())
        # Filtro para archivos de SPSS (.sav) y validación de zona/módulo
        if '.sav' in archivo_norm and zona_busqueda in archivo_norm and clave_norm in archivo_norm:
            # Seguridad: Evita que 'ocupados' atrape accidentalmente 'desocupados'
            if clave_norm == "ocupados" and ("desocupado" in archivo_norm or "deocupado" in archivo_norm):
                continue
            return archivo
    return None

# =====================================================================
# 3. BUCLE PRINCIPAL DE PROCESAMIENTO
# =====================================================================
for nombre_modulo_final, palabra_clave in modulos_claves.items():
    print(f"\n{'-'*60}\nProcesando 2008: {nombre_modulo_final.upper()}\n{'-'*60}")
    datos_modulo = []
    
    for mes_nombre, mes_num in meses.items():
        # Manejo de variantes de nombres de ZIP en 2008
        ruta_zip1 = os.path.join(ruta_base, f"{mes_nombre} (1).zip")
        ruta_zip2 = os.path.join(ruta_base, f"{mes_nombre}.zip")
        ruta_zip = ruta_zip1 if os.path.exists(ruta_zip1) else (ruta_zip2 if os.path.exists(ruta_zip2) else None)
        
        if ruta_zip:
            try:
                with zipfile.ZipFile(ruta_zip, 'r') as z:
                    lista_archivos = z.namelist()
                    for zona_nombre, (zona_num, zona_busqueda) in zonas.items():
                        archivo_objetivo = encontrar_archivo_en_zip(lista_archivos, zona_busqueda, palabra_clave)
                        
                        if archivo_objetivo:
                            tmp_path = ""
                            # Paso 1: Escribir y cerrar el archivo (Solución al ReadstatError)
                            with tempfile.NamedTemporaryFile(delete=False, suffix=".sav") as tmp:
                                tmp.write(z.read(archivo_objetivo))
                                tmp_path = tmp.name
                            
                            # Paso 2: Leer el archivo una vez cerrado
                            try:
                                df, meta = pyreadstat.read_sav(tmp_path, encoding="latin1")
                                df.columns = df.columns.str.lower()
                                df['mes'] = mes_num
                                df['zona_area'] = zona_num
                                
                                # Limpieza profiláctica de variables intrusas del DANE en Ocupados
                                if nombre_modulo_final == "ocupados":
                                    intrusas = ['oficio1', 'ofico2', 'p7530', 'p760', 'rama2d_d', 'rama4d_d']
                                    df = df.drop(columns=[c for c in intrusas if c in df.columns])
                                
                                datos_modulo.append(df)
                                print(f"  [OK] {mes_nombre} - {zona_nombre}")
                            except Exception as e:
                                print(f"  [ERROR] Falló lectura de {mes_nombre}: {e}")
                            finally:
                                if os.path.exists(tmp_path):
                                    os.remove(tmp_path)
            except Exception as e:
                print(f"  [AVISO] No se pudo abrir el zip de {mes_nombre}: {e}")

    # =====================================================================
    # 4. CONSOLIDACIÓN Y GUARDADO
    # =====================================================================
    if datos_modulo:
        print(f"  -> Consolidando {nombre_modulo_final}...")
        df_final = pd.concat(datos_modulo, ignore_index=True)
        
        # Sanitización de columnas tipo 'object' para Stata
        for col in df_final.columns:
            if df_final[col].dtype == 'object':
                df_final[col] = df_final[col].fillna('').astype(str)

        ruta_out = os.path.join(carpeta_salida, f"{nombre_modulo_final}.dta")
        df_final.to_stata(ruta_out, write_index=False, version=118)
        print(f"--> ¡GUARDADO EXITOSO! {len(df_final):,} registros")


------------------------------------------------------------
Procesando 2008: CARACTERISTICAS_GENERALES
------------------------------------------------------------
  [OK] Enero - Cabecera
  [OK] Enero - Resto
  [OK] Febrero - Cabecera
  [OK] Febrero - Resto
  [OK] Marzo - Cabecera
  [OK] Marzo - Resto
  [OK] Abril - Cabecera
  [OK] Abril - Resto
  [OK] Mayo - Cabecera
  [OK] Mayo - Resto
  [OK] Junio - Cabecera
  [OK] Junio - Resto
  [OK] Julio - Cabecera
  [OK] Julio - Resto
  [OK] Agosto - Cabecera
  [OK] Agosto - Resto
  [OK] Septiembre - Cabecera
  [OK] Septiembre - Resto
  [OK] Octubre - Cabecera
  [OK] Octubre - Resto
  [OK] Noviembre - Cabecera
  [OK] Noviembre - Resto
  [OK] Diciembre - Cabecera
  [OK] Diciembre - Resto
  -> Consolidando caracteristicas_generales...
--> ¡GUARDADO EXITOSO! 823,814 registros

------------------------------------------------------------
Procesando 2008: DESOCUPADOS
------------------------------------------------------------
  [OK] Enero - Cabec

# 2009

In [ ]:
import os
import pandas as pd
import unicodedata
import zipfile
import tempfile
import pyreadstat

# =====================================================================
# 1. CONFIGURACIÓN DE RUTAS 2009
# =====================================================================
ruta_base = r"E:\ENTREGA TESIS\Doctorado\Data\2009" 
carpeta_salida = r"E:\ENTREGA TESIS\Doctorado\Data\2009_" 

if not os.path.exists(carpeta_salida):
    os.makedirs(carpeta_salida)
    print(f"Carpeta creada exitosamente en: {carpeta_salida}")

# Estructura de nombres de ZIP para 2009 (01.zip, 02.zip...)
meses_zip = {
    '01': 1, '02': 2, '03': 3, '04': 4, '05': 5, '06': 6, 
    '07': 7, '08': 8, '09': 9, '10': 10, '11': 11, '12': 12
}

# OPCIÓN A: Solo Cabecera y Resto para evitar duplicados de 'Área'
zonas = {
    'Cabecera': (2, 'cabecera'),
    'Resto': (3, 'resto')
}

modulos_claves = {
    "caracteristicas_generales": "caract",
    "desocupados": "desocupados",
    "fuerza_trabajo": "fuerza",
    "inactivos": "inactivos",
    "ocupados": "ocupados", 
    "otras_actividades": "otras",
    "otros_ingresos": "otros ingresos",
    "hogares": "vivienda"
}

# =====================================================================
# 2. FUNCIONES DE APOYO
# =====================================================================
def quitar_tildes(cadena):
    s = ''.join(c for c in unicodedata.normalize('NFD', str(cadena)) if unicodedata.category(c) != 'Mn')
    return s.lower()

def encontrar_archivo_en_zip(lista_archivos_zip, zona_busqueda, palabra_clave):
    clave_norm = quitar_tildes(palabra_clave)
    for archivo in lista_archivos_zip:
        archivo_norm = quitar_tildes(str(archivo).lower())
        # Filtro estricto para archivos SPSS (.sav) y validación de zona/módulo
        if '.sav' in archivo_norm and zona_busqueda in archivo_norm and clave_norm in archivo_norm:
            # Seguridad: Evita que 'ocupados' atrape accidentalmente 'desocupados'
            if clave_norm == "ocupados" and ("desocupado" in archivo_norm or "deocupado" in archivo_norm):
                continue
            return archivo
    return None

# =====================================================================
# 3. BUCLE PRINCIPAL
# =====================================================================
for nombre_modulo_final, palabra_clave in modulos_claves.items():
    print(f"\n{'-'*60}\nProcesando 2009: {nombre_modulo_final.upper()}\n{'-'*60}")
    datos_modulo = []
    
    for mes_str, mes_num in meses_zip.items():
        ruta_zip = os.path.join(ruta_base, f"{mes_str}.zip")
        
        if os.path.exists(ruta_zip):
            try:
                with zipfile.ZipFile(ruta_zip, 'r') as z:
                    lista_archivos = z.namelist()
                    for zona_nombre, (zona_num, zona_busqueda) in zonas.items():
                        archivo_objetivo = encontrar_archivo_en_zip(lista_archivos, zona_busqueda, palabra_clave)
                        
                        if archivo_objetivo:
                            tmp_path = ""
                            # Paso 1: Extraer y CERRAR el archivo temporal (evita ReadstatError)
                            with tempfile.NamedTemporaryFile(delete=False, suffix=".sav") as tmp:
                                tmp.write(z.read(archivo_objetivo))
                                tmp_path = tmp.name
                            
                            # Paso 2: Leer el archivo una vez liberado por el sistema
                            try:
                                df, meta = pyreadstat.read_sav(tmp_path, encoding="latin1")
                                df.columns = df.columns.str.lower()
                                df['mes'] = mes_num
                                df['zona_area'] = zona_num
                                
                                # Limpieza preventiva de variables intrusas identificadas en años anteriores
                                if nombre_modulo_final == "ocupados":
                                    intrusas = ['oficio1', 'ofico2', 'p7530', 'p760', 'rama2d_d', 'rama4d_d']
                                    df = df.drop(columns=[c for c in intrusas if c in df.columns])
                                
                                datos_modulo.append(df)
                                print(f"  [OK] Mes {mes_str} - {zona_nombre}")
                            except Exception as e:
                                print(f"  [ERROR] Falló lectura de Mes {mes_str}: {e}")
                            finally:
                                if os.path.exists(tmp_path):
                                    os.remove(tmp_path)
            except Exception as e:
                print(f"  [AVISO] No se pudo abrir el zip {mes_str}.zip: {e}")

    # =====================================================================
    # 4. CONSOLIDACIÓN FINAL
    # =====================================================================
    if datos_modulo:
        print(f"  -> Consolidando {nombre_modulo_final}...")
        df_final = pd.concat(datos_modulo, ignore_index=True)
        
        # Limpieza de tipos de datos para Stata
        for col in df_final.columns:
            if df_final[col].dtype == 'object':
                df_final[col] = df_final[col].fillna('').astype(str)

        ruta_out = os.path.join(carpeta_salida, f"{nombre_modulo_final}.dta")
        df_final.to_stata(ruta_out, write_index=False, version=118)
        print(f"--> ¡GUARDADO EXITOSO! {len(df_final):,} registros")


------------------------------------------------------------
Procesando 2009: CARACTERISTICAS_GENERALES
------------------------------------------------------------
  [OK] Mes 01 - Cabecera
  [OK] Mes 01 - Resto
  [OK] Mes 02 - Cabecera
  [OK] Mes 02 - Resto
  [OK] Mes 03 - Cabecera
  [OK] Mes 03 - Resto
  [OK] Mes 04 - Cabecera
  [OK] Mes 04 - Resto
  [OK] Mes 05 - Cabecera
  [OK] Mes 05 - Resto
  [OK] Mes 06 - Cabecera
  [OK] Mes 06 - Resto
  [OK] Mes 07 - Cabecera
  [OK] Mes 07 - Resto
  [OK] Mes 08 - Cabecera
  [OK] Mes 08 - Resto
  [OK] Mes 09 - Cabecera
  [OK] Mes 09 - Resto
  [OK] Mes 10 - Cabecera
  [OK] Mes 10 - Resto
  [OK] Mes 11 - Cabecera
  [OK] Mes 11 - Resto
  [OK] Mes 12 - Cabecera
  [OK] Mes 12 - Resto
  -> Consolidando caracteristicas_generales...
--> ¡GUARDADO EXITOSO! 816,242 registros

------------------------------------------------------------
Procesando 2009: DESOCUPADOS
------------------------------------------------------------
  [OK] Mes 01 - Cabecera
  [OK

# 2010

In [15]:
import os
import pandas as pd
import unicodedata
import zipfile
import tempfile
import pyreadstat

# =====================================================================
# 1. CONFIGURACIÓN DE RUTAS
# =====================================================================
ruta_base = r"E:\ENTREGA TESIS\Doctorado\Data\2010\GEIH_Empalme_2010" 
carpeta_salida = r"E:\ENTREGA TESIS\Doctorado\Data\2010_" 

if not os.path.exists(carpeta_salida):
    os.makedirs(carpeta_salida)

meses = {
    '1. Enero': 1, '2. Febrero': 2, '3. Marzo': 3, '4. Abril': 4, 
    '5. Mayo': 5, '6. Junio': 6, '7. Julio': 7, '8. Agosto': 8, 
    '9. Septiembre': 9, '10. Octubre': 10, '11. Noviembre': 11, '12. Diciembre': 12
}

# Definimos las claves. Para desocupados usamos una lista por el error del DANE
modulos_claves = {
    "caracteristicas_generales": ["caract"],
    "desocupados": ["desocupado", "deocupado", "descupado"], # <--- Aquí corregimos el error de Mayo y Octubre
    "fuerza_trabajo": ["fuerza"],
    "inactivos": ["inactivos"],
    "ocupados": ["ocupados"], 
    "otras_actividades": ["otras"],
    "otros_ingresos": ["otros ingresos"],
    "hogares": ["vivienda"]
}

def quitar_tildes(cadena):
    try:
        s = ''.join(c for c in unicodedata.normalize('NFD', str(cadena)) if unicodedata.category(c) != 'Mn')
        return s.lower()
    except:
        return str(cadena).lower()

def encontrar_archivo_en_zip(lista_archivos_zip, lista_palabras_clave):
    for archivo in lista_archivos_zip:
        archivo_norm = quitar_tildes(archivo.lower())
        if '.dta' in archivo_norm:
            for palabra in lista_palabras_clave:
                palabra_norm = quitar_tildes(palabra)
                if palabra_norm in archivo_norm:
                    # Regla de exclusión: si busco ocupados, que no traiga desocupados
                    if palabra_norm == "ocupados" and ("desocupado" in archivo_norm or "deocupado" in archivo_norm):
                        continue
                    return archivo
    return None

# =====================================================================
# 2. PROCESAMIENTO
# =====================================================================
diccionario_variables = []

for nombre_modulo_final, palabras_clave in modulos_claves.items():
    print(f"\nProcesando Módulo: {nombre_modulo_final.upper()}")
    datos_modulo = []
    
    for mes_nombre, mes_num in meses.items():
        ruta_zip = os.path.join(ruta_base, f"{mes_nombre}.zip")
        if os.path.exists(ruta_zip):
            with zipfile.ZipFile(ruta_zip, 'r') as z:
                archivo_objetivo = encontrar_archivo_en_zip(z.namelist(), palabras_clave)
                
                if archivo_objetivo:
                    tmp_path = ""
                    try:
                        with tempfile.NamedTemporaryFile(delete=False, suffix=".dta") as tmp:
                            tmp.write(z.read(archivo_objetivo))
                            tmp_path = tmp.name
                        
                        df, meta = pyreadstat.read_dta(tmp_path)
                        df.columns = df.columns.str.lower()
                        df['mes'] = mes_num
                        datos_modulo.append(df)
                        print(f"  [OK] {mes_nombre}")
                    except Exception as e:
                        print(f"  [ERROR] {mes_nombre}: {e}")
                    finally:
                        if tmp_path and os.path.exists(tmp_path): os.remove(tmp_path)
                else:
                    print(f"  [AVISO] No encontrado en {mes_nombre}")

    if datos_modulo:
        df_final = pd.concat(datos_modulo, ignore_index=True)
        # Limpieza rápida para Stata
        for col in df_final.columns:
            if df_final[col].dtype == 'object':
                df_final[col] = df_final[col].astype(str).replace('nan', '')
        
        df_final.to_stata(os.path.join(carpeta_salida, f"{nombre_modulo_final}.dta"), write_index=False, version=118)
        for var in df_final.columns:
            diccionario_variables.append({"Módulo": nombre_modulo_final, "Variable": var})

pd.DataFrame(diccionario_variables).drop_duplicates().to_excel(os.path.join(carpeta_salida, "Diccionario_2010.xlsx"), index=False)
print("\n¡2010 COMPLETO!")


Procesando Módulo: CARACTERISTICAS_GENERALES
  [OK] 1. Enero
  [OK] 2. Febrero
  [OK] 3. Marzo
  [OK] 4. Abril
  [OK] 5. Mayo
  [OK] 6. Junio
  [OK] 7. Julio
  [OK] 8. Agosto
  [OK] 9. Septiembre
  [OK] 10. Octubre
  [OK] 11. Noviembre
  [OK] 12. Diciembre

Procesando Módulo: DESOCUPADOS
  [OK] 1. Enero
  [OK] 2. Febrero
  [OK] 3. Marzo
  [OK] 4. Abril
  [OK] 5. Mayo
  [OK] 6. Junio
  [OK] 7. Julio
  [OK] 8. Agosto
  [OK] 9. Septiembre
  [OK] 10. Octubre
  [OK] 11. Noviembre
  [OK] 12. Diciembre

Procesando Módulo: FUERZA_TRABAJO
  [OK] 1. Enero
  [OK] 2. Febrero
  [OK] 3. Marzo
  [OK] 4. Abril
  [OK] 5. Mayo
  [OK] 6. Junio
  [OK] 7. Julio
  [OK] 8. Agosto
  [OK] 9. Septiembre
  [OK] 10. Octubre
  [OK] 11. Noviembre
  [OK] 12. Diciembre

Procesando Módulo: INACTIVOS
  [OK] 1. Enero
  [OK] 2. Febrero
  [OK] 3. Marzo
  [OK] 4. Abril
  [OK] 5. Mayo
  [OK] 6. Junio
  [OK] 7. Julio
  [OK] 8. Agosto
  [OK] 9. Septiembre
  [OK] 10. Octubre
  [OK] 11. Noviembre
  [OK] 12. Diciembre

Procesan

# 2011

In [1]:
import os
import pandas as pd
import unicodedata
import zipfile
import tempfile
import pyreadstat

# =====================================================================
# 1. CONFIGURACIÓN DE RUTAS (Actualizado para 2011)
# =====================================================================
# Ruta donde están tus 12 zips de 2011
ruta_base = r"E:\ENTREGA TESIS\Doctorado\Data\2011\GEIH_Empalme_2011" 
# Carpeta donde se guardarán los .dta finales de 2011
carpeta_salida = r"E:\ENTREGA TESIS\Doctorado\Data\2011_" 

if not os.path.exists(carpeta_salida):
    os.makedirs(carpeta_salida)
    print(f"Carpeta creada exitosamente en: {carpeta_salida}")

# Diccionario de meses para asignar la variable 'mes'
meses = {
    '1. Enero': 1, '2. Febrero': 2, '3. Marzo': 3, '4. Abril': 4, 
    '5. Mayo': 5, '6. Junio': 6, '7. Julio': 7, '8. Agosto': 8, 
    '9. Septiembre': 9, '10. Octubre': 10, '11. Noviembre': 11, '12. Diciembre': 12
}

# Claves de búsqueda ajustadas según lo visto en Febrero 2011
modulos_claves = {
    "caracteristicas_generales": ["caract"],
    "desocupados": ["desocupado", "deocupado", "descupado"], 
    "fuerza_trabajo": ["fuerza"],
    "inactivos": ["inactivos"],
    "ocupados": ["ocupados"], 
    "otras_actividades": ["otras"],
    "otros_ingresos": ["otros ingresos"],
    "hogares": ["vivienda", "hogar"]
}

# =====================================================================
# 2. FUNCIONES DE APOYO
# =====================================================================
def quitar_tildes(cadena):
    try:
        s = ''.join(c for c in unicodedata.normalize('NFD', str(cadena)) if unicodedata.category(c) != 'Mn')
        return s.lower()
    except:
        return str(cadena).lower()

def encontrar_archivo_en_zip(lista_archivos_zip, lista_palabras_clave):
    for archivo in lista_archivos_zip:
        archivo_norm = quitar_tildes(archivo.lower())
        # Filtramos solo archivos de datos Stata (.dta o .DTA)
        if '.dta' in archivo_norm:
            for palabra in lista_palabras_clave:
                palabra_norm = quitar_tildes(palabra)
                if palabra_norm in archivo_norm:
                    # REGLA DE EXCLUSIÓN: Evitar que 'ocupados' atrape 'desocupados'
                    if palabra_norm == "ocupados" and ("desocupado" in archivo_norm or "deocupado" in archivo_norm):
                        continue
                    return archivo
    return None

# =====================================================================
# 3. BUCLE PRINCIPAL DE PROCESAMIENTO
# =====================================================================
diccionario_variables = []

for nombre_modulo_final, palabras_clave in modulos_claves.items():
    print(f"\n{'-'*60}\nProcesando Módulo: {nombre_modulo_final.upper()}\n{'-'*60}")
    datos_modulo = []
    
    for mes_nombre, mes_num in meses.items():
        ruta_zip = os.path.join(ruta_base, f"{mes_nombre}.zip")
        
        if os.path.exists(ruta_zip):
            try:
                with zipfile.ZipFile(ruta_zip, 'r') as z:
                    archivo_objetivo = encontrar_archivo_en_zip(z.namelist(), palabras_clave)
                    
                    if archivo_objetivo:
                        tmp_path = ""
                        try:
                            # Extraer a archivo temporal para lectura rápida
                            with tempfile.NamedTemporaryFile(delete=False, suffix=".dta") as tmp:
                                tmp.write(z.read(archivo_objetivo))
                                tmp_path = tmp.name
                            
                            # Cargar datos
                            df, meta = pyreadstat.read_dta(tmp_path)
                            
                            # Estandarizar columnas a minúsculas
                            df.columns = df.columns.str.lower()
                            df = df.copy() 
                            df['mes'] = mes_num
                            
                            datos_modulo.append(df)
                            print(f"  [OK] {mes_nombre}")
                            
                        except Exception as e:
                            print(f"  [ERROR] Al leer {mes_nombre}: {e}")
                        finally:
                            if tmp_path and os.path.exists(tmp_path):
                                os.remove(tmp_path)
                    else:
                        print(f"  [AVISO] No se encontró el archivo para '{nombre_modulo_final}' en {mes_nombre}")
            except Exception as e:
                print(f"  [ERROR GRAVE] No se pudo abrir el zip {mes_nombre}: {e}")
        else:
            print(f"  [AVISO] No existe el archivo zip: {mes_nombre}")

    # =====================================================================
    # 4. CONSOLIDACIÓN Y EXPORTACIÓN
    # =====================================================================
    if datos_modulo:
        print(f"  -> Consolidando 12 meses de {nombre_modulo_final}...")
        df_final = pd.concat(datos_modulo, ignore_index=True)
        
        # Limpieza de tipos de datos para Stata
        for col in df_final.columns:
            if df_final[col].dtype == 'object':
                df_final[col] = df_final[col].astype(str).replace('nan', '')

        ruta_archivo_final = os.path.join(carpeta_salida, f"{nombre_modulo_final}.dta")
        df_final.to_stata(ruta_archivo_final, write_index=False, version=118)
        print(f"--> ¡GUARDADO EXITOSO!: {nombre_modulo_final}.dta")
        
        # Registrar variables para el diccionario final
        for var in df_final.columns:
            diccionario_variables.append({"Módulo": nombre_modulo_final, "Variable": var})
    else:
        print(f"--> ADVERTENCIA: El módulo {nombre_modulo_final} no generó datos.")

# =====================================================================
# 5. GENERACIÓN DEL DICCIONARIO EN EXCEL
# =====================================================================
if diccionario_variables:
    print("\nGenerando diccionario de variables 2011...")
    df_dicc = pd.DataFrame(diccionario_variables).drop_duplicates().sort_values(by=["Módulo", "Variable"])
    ruta_excel = os.path.join(carpeta_salida, "Diccionario_Variables_2011.xlsx")
    df_dicc.to_excel(ruta_excel, index=False)
    print(f"¡PROCESO COMPLETADO! Excel guardado en: {ruta_excel}")

Carpeta creada exitosamente en: E:\ENTREGA TESIS\Doctorado\Data\2011_

------------------------------------------------------------
Procesando Módulo: CARACTERISTICAS_GENERALES
------------------------------------------------------------
  [OK] 1. Enero
  [OK] 2. Febrero
  [OK] 3. Marzo
  [OK] 4. Abril
  [OK] 5. Mayo
  [OK] 6. Junio
  [OK] 7. Julio
  [OK] 8. Agosto
  [OK] 9. Septiembre
  [OK] 10. Octubre
  [OK] 11. Noviembre
  [OK] 12. Diciembre
  -> Consolidando 12 meses de caracteristicas_generales...
--> ¡GUARDADO EXITOSO!: caracteristicas_generales.dta

------------------------------------------------------------
Procesando Módulo: DESOCUPADOS
------------------------------------------------------------
  [OK] 1. Enero
  [OK] 2. Febrero
  [OK] 3. Marzo
  [OK] 4. Abril
  [OK] 5. Mayo
  [OK] 6. Junio
  [OK] 7. Julio
  [OK] 8. Agosto
  [OK] 9. Septiembre
  [OK] 10. Octubre
  [OK] 11. Noviembre
  [OK] 12. Diciembre
  -> Consolidando 12 meses de desocupados...
--> ¡GUARDADO EXITOSO!: de

# 2012 

In [2]:
import os
import pandas as pd
import unicodedata
import zipfile
import tempfile
import pyreadstat

# =====================================================================
# 1. CONFIGURACIÓN DE RUTAS (Actualizado para 2012)
# =====================================================================
ruta_base = r"E:\ENTREGA TESIS\Doctorado\Data\2012\GEIH_Empalme_2012" 
carpeta_salida = r"E:\ENTREGA TESIS\Doctorado\Data\2012_" 

if not os.path.exists(carpeta_salida):
    os.makedirs(carpeta_salida)

meses = {
    '1. Enero': 1, '2. Febrero': 2, '3. Marzo': 3, '4. Abril': 4, 
    '5. Mayo': 5, '6. Junio': 6, '7. Julio': 7, '8. Agosto': 8, 
    '9. Septiembre': 9, '10. Octubre': 10, '11. Noviembre': 11, '12. Diciembre': 12
}

# Claves de búsqueda robustas
modulos_claves = {
    "caracteristicas_generales": ["caract"],
    "desocupados": ["desocupado", "deocupado", "descupado"], 
    "fuerza_trabajo": ["fuerza"],
    "inactivos": ["inactivos"],
    "ocupados": ["ocupados"], 
    "otras_actividades": ["otras"],
    "otros_ingresos": ["otros ingresos"],
    "hogares": ["vivienda", "hogar"]
}

def quitar_tildes(cadena):
    try:
        s = ''.join(c for c in unicodedata.normalize('NFD', str(cadena)) if unicodedata.category(c) != 'Mn')
        return s.lower()
    except:
        return str(cadena).lower()

def encontrar_archivo_en_zip(lista_archivos_zip, lista_palabras_clave):
    for archivo in lista_archivos_zip:
        archivo_norm = quitar_tildes(archivo.lower())
        if '.dta' in archivo_norm:
            for palabra in lista_palabras_clave:
                palabra_norm = quitar_tildes(palabra)
                if palabra_norm in archivo_norm:
                    # Protección para no mezclar ocupados con desocupados
                    if palabra_norm == "ocupados" and ("desocupado" in archivo_norm or "deocupado" in archivo_norm):
                        continue
                    return archivo
    return None

# =====================================================================
# 2. PROCESAMIENTO
# =====================================================================
diccionario_variables = []

for nombre_modulo_final, palabras_clave in modulos_claves.items():
    print(f"\nProcesando Módulo 2012: {nombre_modulo_final.upper()}")
    datos_modulo = []
    
    for mes_nombre, mes_num in meses.items():
        ruta_zip = os.path.join(ruta_base, f"{mes_nombre}.zip")
        if os.path.exists(ruta_zip):
            with zipfile.ZipFile(ruta_zip, 'r') as z:
                archivo_objetivo = encontrar_archivo_en_zip(z.namelist(), palabras_clave)
                
                if archivo_objetivo:
                    tmp_path = ""
                    try:
                        with tempfile.NamedTemporaryFile(delete=False, suffix=".dta") as tmp:
                            tmp.write(z.read(archivo_objetivo))
                            tmp_path = tmp.name
                        
                        df, meta = pyreadstat.read_dta(tmp_path)
                        df.columns = df.columns.str.lower()
                        df['mes'] = mes_num
                        datos_modulo.append(df)
                        print(f"  [OK] {mes_nombre}")
                    except Exception as e:
                        print(f"  [ERROR] {mes_nombre}: {e}")
                    finally:
                        if tmp_path and os.path.exists(tmp_path): os.remove(tmp_path)
                else:
                    print(f"  [AVISO] No encontrado en {mes_nombre}")

    if datos_modulo:
        df_final = pd.concat(datos_modulo, ignore_index=True)
        # Limpieza para Stata
        for col in df_final.columns:
            if df_final[col].dtype == 'object':
                df_final[col] = df_final[col].astype(str).replace('nan', '')
        
        df_final.to_stata(os.path.join(carpeta_salida, f"{nombre_modulo_final}.dta"), write_index=False, version=118)
        for var in df_final.columns:
            diccionario_variables.append({"Módulo": nombre_modulo_final, "Variable": var})

# Guardar Diccionario 2012
pd.DataFrame(diccionario_variables).drop_duplicates().to_excel(os.path.join(carpeta_salida, "Diccionario_2012.xlsx"), index=False)
print("\n¡PROCESO 2012 COMPLETADO!")


Procesando Módulo 2012: CARACTERISTICAS_GENERALES
  [OK] 1. Enero
  [OK] 2. Febrero
  [OK] 3. Marzo
  [OK] 4. Abril
  [OK] 5. Mayo
  [OK] 6. Junio
  [OK] 7. Julio
  [OK] 8. Agosto
  [OK] 9. Septiembre
  [OK] 10. Octubre
  [OK] 11. Noviembre
  [OK] 12. Diciembre

Procesando Módulo 2012: DESOCUPADOS
  [OK] 1. Enero
  [OK] 2. Febrero
  [OK] 3. Marzo
  [OK] 4. Abril
  [OK] 5. Mayo
  [OK] 6. Junio
  [OK] 7. Julio
  [OK] 8. Agosto
  [OK] 9. Septiembre
  [OK] 10. Octubre
  [OK] 11. Noviembre
  [OK] 12. Diciembre

Procesando Módulo 2012: FUERZA_TRABAJO
  [OK] 1. Enero
  [OK] 2. Febrero
  [OK] 3. Marzo
  [OK] 4. Abril
  [OK] 5. Mayo
  [OK] 6. Junio
  [OK] 7. Julio
  [OK] 8. Agosto
  [OK] 9. Septiembre
  [OK] 10. Octubre
  [OK] 11. Noviembre
  [OK] 12. Diciembre

Procesando Módulo 2012: INACTIVOS
  [OK] 1. Enero
  [OK] 2. Febrero
  [OK] 3. Marzo
  [OK] 4. Abril
  [OK] 5. Mayo
  [OK] 6. Junio
  [OK] 7. Julio
  [OK] 8. Agosto
  [OK] 9. Septiembre
  [OK] 10. Octubre
  [OK] 11. Noviembre
  [OK] 12.

# 2013

In [3]:
import os
import pandas as pd
import unicodedata
import zipfile
import tempfile
import pyreadstat

# =====================================================================
# 1. CONFIGURACIÓN DE RUTAS (Ajustado a la ruta proporcionada)
# =====================================================================
# Nota: Se usa 'GEIH_Emplame_2013' tal como aparece en tu mensaje
ruta_base = r"E:\ENTREGA TESIS\Doctorado\Data\2013\GEIH_Emplame_2013" 
carpeta_salida = r"E:\ENTREGA TESIS\Doctorado\Data\2013_" 

if not os.path.exists(carpeta_salida):
    os.makedirs(carpeta_salida)

meses = {
    '1. Enero': 1, '2. Febrero': 2, '3. Marzo': 3, '4. Abril': 4, 
    '5. Mayo': 5, '6. Junio': 6, '7. Julio': 7, '8. Agosto': 8, 
    '9. Septiembre': 9, '10. Octubre': 10, '11. Noviembre': 11, '12. Diciembre': 12
}

# Claves de búsqueda robustas para capturar variaciones ortográficas
modulos_claves = {
    "caracteristicas_generales": ["caract"],
    "desocupados": ["desocupado", "deocupado", "descupado"], 
    "fuerza_trabajo": ["fuerza"],
    "inactivos": ["inactivos"],
    "ocupados": ["ocupados"], 
    "otras_actividades": ["otras"],
    "otros_ingresos": ["otros ingresos"],
    "hogares": ["vivienda", "hogar"]
}

def quitar_tildes(cadena):
    try:
        s = ''.join(c for c in unicodedata.normalize('NFD', str(cadena)) if unicodedata.category(c) != 'Mn')
        return s.lower()
    except:
        return str(cadena).lower()

def encontrar_archivo_en_zip(lista_archivos_zip, lista_palabras_clave):
    for archivo in lista_archivos_zip:
        archivo_norm = quitar_tildes(archivo.lower())
        if '.dta' in archivo_norm:
            for palabra in lista_palabras_clave:
                palabra_norm = quitar_tildes(palabra)
                if palabra_norm in archivo_norm:
                    # Protección para no mezclar ocupados con desocupados
                    if palabra_norm == "ocupados" and ("desocupado" in archivo_norm or "deocupado" in archivo_norm):
                        continue
                    return archivo
    return None

# =====================================================================
# 2. PROCESAMIENTO
# =====================================================================
diccionario_variables = []

for nombre_modulo_final, palabras_clave in modulos_claves.items():
    print(f"\nProcesando Módulo 2013: {nombre_modulo_final.upper()}")
    datos_modulo = []
    
    for mes_nombre, mes_num in meses.items():
        ruta_zip = os.path.join(ruta_base, f"{mes_nombre}.zip")
        if os.path.exists(ruta_zip):
            with zipfile.ZipFile(ruta_zip, 'r') as z:
                archivo_objetivo = encontrar_archivo_en_zip(z.namelist(), palabras_clave)
                
                if archivo_objetivo:
                    tmp_path = ""
                    try:
                        with tempfile.NamedTemporaryFile(delete=False, suffix=".dta") as tmp:
                            tmp.write(z.read(archivo_objetivo))
                            tmp_path = tmp.name
                        
                        df, meta = pyreadstat.read_dta(tmp_path)
                        df.columns = df.columns.str.lower()
                        df['mes'] = mes_num
                        datos_modulo.append(df)
                        print(f"  [OK] {mes_nombre}")
                    except Exception as e:
                        print(f"  [ERROR] {mes_nombre}: {e}")
                    finally:
                        if tmp_path and os.path.exists(tmp_path): os.remove(tmp_path)
                else:
                    print(f"  [AVISO] No encontrado en {mes_nombre}")
        else:
            print(f"  [AVISO] No existe el zip: {mes_nombre}")

    if datos_modulo:
        df_final = pd.concat(datos_modulo, ignore_index=True)
        # Limpieza para Stata
        for col in df_final.columns:
            if df_final[col].dtype == 'object':
                df_final[col] = df_final[col].astype(str).replace('nan', '')
        
        df_final.to_stata(os.path.join(carpeta_salida, f"{nombre_modulo_final}.dta"), write_index=False, version=118)
        for var in df_final.columns:
            diccionario_variables.append({"Módulo": nombre_modulo_final, "Variable": var})

# Guardar Diccionario 2013
pd.DataFrame(diccionario_variables).drop_duplicates().to_excel(os.path.join(carpeta_salida, "Diccionario_2013.xlsx"), index=False)
print("\n¡PROCESO 2013 COMPLETADO!")


Procesando Módulo 2013: CARACTERISTICAS_GENERALES
  [OK] 1. Enero
  [OK] 2. Febrero
  [OK] 3. Marzo
  [OK] 4. Abril
  [OK] 5. Mayo
  [OK] 6. Junio
  [OK] 7. Julio
  [OK] 8. Agosto
  [OK] 9. Septiembre
  [OK] 10. Octubre
  [OK] 11. Noviembre
  [OK] 12. Diciembre

Procesando Módulo 2013: DESOCUPADOS
  [OK] 1. Enero
  [OK] 2. Febrero
  [OK] 3. Marzo
  [OK] 4. Abril
  [OK] 5. Mayo
  [OK] 6. Junio
  [OK] 7. Julio
  [OK] 8. Agosto
  [OK] 9. Septiembre
  [OK] 10. Octubre
  [OK] 11. Noviembre
  [OK] 12. Diciembre

Procesando Módulo 2013: FUERZA_TRABAJO
  [OK] 1. Enero
  [OK] 2. Febrero
  [OK] 3. Marzo
  [OK] 4. Abril
  [OK] 5. Mayo
  [OK] 6. Junio
  [OK] 7. Julio
  [OK] 8. Agosto
  [OK] 9. Septiembre
  [OK] 10. Octubre
  [OK] 11. Noviembre
  [OK] 12. Diciembre

Procesando Módulo 2013: INACTIVOS
  [OK] 1. Enero
  [OK] 2. Febrero
  [OK] 3. Marzo
  [OK] 4. Abril
  [OK] 5. Mayo
  [OK] 6. Junio
  [OK] 7. Julio
  [OK] 8. Agosto
  [OK] 9. Septiembre
  [OK] 10. Octubre
  [OK] 11. Noviembre
  [OK] 12.

# 2014

In [4]:
import os
import pandas as pd
import unicodedata
import zipfile
import tempfile
import pyreadstat

# =====================================================================
# 1. CONFIGURACIÓN DE RUTAS (Actualizado para 2014)
# =====================================================================
ruta_base = r"E:\ENTREGA TESIS\Doctorado\Data\2014\GEIH_Empalme_2014" 
carpeta_salida = r"E:\ENTREGA TESIS\Doctorado\Data\2014_" 

if not os.path.exists(carpeta_salida):
    os.makedirs(carpeta_salida)

meses = {
    '1. Enero': 1, '2. Febrero': 2, '3. Marzo': 3, '4. Abril': 4, 
    '5. Mayo': 5, '6. Junio': 6, '7. Julio': 7, '8. Agosto': 8, 
    '9. Septiembre': 9, '10. Octubre': 10, '11. Noviembre': 11, '12. Diciembre': 12
}

# Claves de búsqueda robustas
modulos_claves = {
    "caracteristicas_generales": ["caract"],
    "desocupados": ["desocupado", "deocupado", "descupado"], 
    "fuerza_trabajo": ["fuerza"],
    "inactivos": ["inactivos"],
    "ocupados": ["ocupados"], 
    "otras_actividades": ["otras"],
    "otros_ingresos": ["otros ingresos"],
    "hogares": ["vivienda", "hogar"]
}

def quitar_tildes(cadena):
    try:
        s = ''.join(c for c in unicodedata.normalize('NFD', str(cadena)) if unicodedata.category(c) != 'Mn')
        return s.lower()
    except:
        return str(cadena).lower()

def encontrar_archivo_en_zip(lista_archivos_zip, lista_palabras_clave):
    for archivo in lista_archivos_zip:
        archivo_norm = quitar_tildes(archivo.lower())
        if '.dta' in archivo_norm:
            for palabra in lista_palabras_clave:
                palabra_norm = quitar_tildes(palabra)
                if palabra_norm in archivo_norm:
                    # Protección para no mezclar ocupados con desocupados
                    if palabra_norm == "ocupados" and ("desocupado" in archivo_norm or "deocupado" in archivo_norm):
                        continue
                    return archivo
    return None

# =====================================================================
# 2. PROCESAMIENTO
# =====================================================================
diccionario_variables = []

for nombre_modulo_final, palabras_clave in modulos_claves.items():
    print(f"\nProcesando Módulo 2014: {nombre_modulo_final.upper()}")
    datos_modulo = []
    
    for mes_nombre, mes_num in meses.items():
        ruta_zip = os.path.join(ruta_base, f"{mes_nombre}.zip")
        if os.path.exists(ruta_zip):
            with zipfile.ZipFile(ruta_zip, 'r') as z:
                archivo_objetivo = encontrar_archivo_en_zip(z.namelist(), palabras_clave)
                
                if archivo_objetivo:
                    tmp_path = ""
                    try:
                        with tempfile.NamedTemporaryFile(delete=False, suffix=".dta") as tmp:
                            tmp.write(z.read(archivo_objetivo))
                            tmp_path = tmp.name
                        
                        df, meta = pyreadstat.read_dta(tmp_path)
                        df.columns = df.columns.str.lower()
                        df['mes'] = mes_num
                        datos_modulo.append(df)
                        print(f"  [OK] {mes_nombre}")
                    except Exception as e:
                        print(f"  [ERROR] {mes_nombre}: {e}")
                    finally:
                        if tmp_path and os.path.exists(tmp_path): os.remove(tmp_path)
                else:
                    print(f"  [AVISO] No encontrado en {mes_nombre}")

    if datos_modulo:
        df_final = pd.concat(datos_modulo, ignore_index=True)
        # Limpieza para Stata
        for col in df_final.columns:
            if df_final[col].dtype == 'object':
                df_final[col] = df_final[col].astype(str).replace('nan', '')
        
        df_final.to_stata(os.path.join(carpeta_salida, f"{nombre_modulo_final}.dta"), write_index=False, version=118)
        for var in df_final.columns:
            diccionario_variables.append({"Módulo": nombre_modulo_final, "Variable": var})

# Guardar Diccionario 2014
pd.DataFrame(diccionario_variables).drop_duplicates().to_excel(os.path.join(carpeta_salida, "Diccionario_2014.xlsx"), index=False)
print("\n¡PROCESO 2014 COMPLETADO!")


Procesando Módulo 2014: CARACTERISTICAS_GENERALES
  [OK] 1. Enero
  [OK] 2. Febrero
  [OK] 3. Marzo
  [OK] 4. Abril
  [OK] 5. Mayo
  [OK] 6. Junio
  [OK] 7. Julio
  [OK] 8. Agosto
  [OK] 9. Septiembre
  [OK] 10. Octubre
  [OK] 11. Noviembre
  [OK] 12. Diciembre

Procesando Módulo 2014: DESOCUPADOS
  [OK] 1. Enero
  [OK] 2. Febrero
  [OK] 3. Marzo
  [OK] 4. Abril
  [OK] 5. Mayo
  [OK] 6. Junio
  [OK] 7. Julio
  [OK] 8. Agosto
  [OK] 9. Septiembre
  [OK] 10. Octubre
  [OK] 11. Noviembre
  [OK] 12. Diciembre

Procesando Módulo 2014: FUERZA_TRABAJO
  [OK] 1. Enero
  [OK] 2. Febrero
  [OK] 3. Marzo
  [OK] 4. Abril
  [OK] 5. Mayo
  [OK] 6. Junio
  [OK] 7. Julio
  [OK] 8. Agosto
  [OK] 9. Septiembre
  [OK] 10. Octubre
  [OK] 11. Noviembre
  [OK] 12. Diciembre

Procesando Módulo 2014: INACTIVOS
  [OK] 1. Enero
  [OK] 2. Febrero
  [OK] 3. Marzo
  [OK] 4. Abril
  [OK] 5. Mayo
  [OK] 6. Junio
  [OK] 7. Julio
  [OK] 8. Agosto
  [OK] 9. Septiembre
  [OK] 10. Octubre
  [OK] 11. Noviembre
  [OK] 12.

# 2015

In [5]:
import os
import pandas as pd
import unicodedata
import zipfile
import tempfile
import pyreadstat

# =====================================================================
# 1. CONFIGURACIÓN DE RUTAS (Actualizado para 2015)
# =====================================================================
ruta_base = r"E:\ENTREGA TESIS\Doctorado\Data\2015\GEIH_Empalme_2015" 
carpeta_salida = r"E:\ENTREGA TESIS\Doctorado\Data\2015_" 

if not os.path.exists(carpeta_salida):
    os.makedirs(carpeta_salida)

meses = {
    '1. Enero': 1, '2. Febrero': 2, '3. Marzo': 3, '4. Abril': 4, 
    '5. Mayo': 5, '6. Junio': 6, '7. Julio': 7, '8. Agosto': 8, 
    '9. Septiembre': 9, '10. Octubre': 10, '11. Noviembre': 11, '12. Diciembre': 12
}

# Claves de búsqueda robustas
modulos_claves = {
    "caracteristicas_generales": ["caract"],
    "desocupados": ["desocupado", "deocupado", "descupado"], 
    "fuerza_trabajo": ["fuerza"],
    "inactivos": ["inactivos"],
    "ocupados": ["ocupados"], 
    "otras_actividades": ["otras"],
    "otros_ingresos": ["otros ingresos"],
    "hogares": ["vivienda", "hogar"]
}

def quitar_tildes(cadena):
    try:
        s = ''.join(c for c in unicodedata.normalize('NFD', str(cadena)) if unicodedata.category(c) != 'Mn')
        return s.lower()
    except:
        return str(cadena).lower()

def encontrar_archivo_en_zip(lista_archivos_zip, lista_palabras_clave):
    for archivo in lista_archivos_zip:
        archivo_norm = quitar_tildes(archivo.lower())
        if '.dta' in archivo_norm:
            for palabra in lista_palabras_clave:
                palabra_norm = quitar_tildes(palabra)
                if palabra_norm in archivo_norm:
                    # Protección para no mezclar ocupados con desocupados
                    if palabra_norm == "ocupados" and ("desocupado" in archivo_norm or "deocupado" in archivo_norm):
                        continue
                    return archivo
    return None

# =====================================================================
# 2. PROCESAMIENTO
# =====================================================================
diccionario_variables = []

for nombre_modulo_final, palabras_clave in modulos_claves.items():
    print(f"\nProcesando Módulo 2015: {nombre_modulo_final.upper()}")
    datos_modulo = []
    
    for mes_nombre, mes_num in meses.items():
        ruta_zip = os.path.join(ruta_base, f"{mes_nombre}.zip")
        if os.path.exists(ruta_zip):
            with zipfile.ZipFile(ruta_zip, 'r') as z:
                archivo_objetivo = encontrar_archivo_en_zip(z.namelist(), palabras_clave)
                
                if archivo_objetivo:
                    tmp_path = ""
                    try:
                        with tempfile.NamedTemporaryFile(delete=False, suffix=".dta") as tmp:
                            tmp.write(z.read(archivo_objetivo))
                            tmp_path = tmp.name
                        
                        df, meta = pyreadstat.read_dta(tmp_path)
                        df.columns = df.columns.str.lower()
                        df['mes'] = mes_num
                        datos_modulo.append(df)
                        print(f"  [OK] {mes_nombre}")
                    except Exception as e:
                        print(f"  [ERROR] {mes_nombre}: {e}")
                    finally:
                        if tmp_path and os.path.exists(tmp_path): os.remove(tmp_path)
                else:
                    print(f"  [AVISO] No encontrado en {mes_nombre}")

    if datos_modulo:
        df_final = pd.concat(datos_modulo, ignore_index=True)
        # Limpieza para Stata
        for col in df_final.columns:
            if df_final[col].dtype == 'object':
                df_final[col] = df_final[col].astype(str).replace('nan', '')
        
        df_final.to_stata(os.path.join(carpeta_salida, f"{nombre_modulo_final}.dta"), write_index=False, version=118)
        for var in df_final.columns:
            diccionario_variables.append({"Módulo": nombre_modulo_final, "Variable": var})

# Guardar Diccionario 2015
pd.DataFrame(diccionario_variables).drop_duplicates().to_excel(os.path.join(carpeta_salida, "Diccionario_2015.xlsx"), index=False)
print("\n¡PROCESO 2015 COMPLETADO!")


Procesando Módulo 2015: CARACTERISTICAS_GENERALES
  [OK] 1. Enero
  [OK] 2. Febrero
  [OK] 3. Marzo
  [OK] 4. Abril
  [OK] 5. Mayo
  [OK] 6. Junio
  [OK] 7. Julio
  [OK] 8. Agosto
  [OK] 9. Septiembre
  [OK] 10. Octubre
  [OK] 11. Noviembre
  [OK] 12. Diciembre

Procesando Módulo 2015: DESOCUPADOS
  [OK] 1. Enero
  [OK] 2. Febrero
  [OK] 3. Marzo
  [OK] 4. Abril
  [OK] 5. Mayo
  [OK] 6. Junio
  [OK] 7. Julio
  [OK] 8. Agosto
  [OK] 9. Septiembre
  [OK] 10. Octubre
  [OK] 11. Noviembre
  [OK] 12. Diciembre

Procesando Módulo 2015: FUERZA_TRABAJO
  [OK] 1. Enero
  [OK] 2. Febrero
  [OK] 3. Marzo
  [OK] 4. Abril
  [OK] 5. Mayo
  [OK] 6. Junio
  [OK] 7. Julio
  [OK] 8. Agosto
  [OK] 9. Septiembre
  [OK] 10. Octubre
  [OK] 11. Noviembre
  [OK] 12. Diciembre

Procesando Módulo 2015: INACTIVOS
  [OK] 1. Enero
  [OK] 2. Febrero
  [OK] 3. Marzo
  [OK] 4. Abril
  [OK] 5. Mayo
  [OK] 6. Junio
  [OK] 7. Julio
  [OK] 8. Agosto
  [OK] 9. Septiembre
  [OK] 10. Octubre
  [OK] 11. Noviembre
  [OK] 12.

# 2016

In [6]:
import os
import pandas as pd
import unicodedata
import zipfile
import tempfile
import pyreadstat

# =====================================================================
# 1. CONFIGURACIÓN DE RUTAS (Actualizado para 2016)
# =====================================================================
ruta_base = r"E:\ENTREGA TESIS\Doctorado\Data\2016\GEIH_Empalme_2016" 
carpeta_salida = r"E:\ENTREGA TESIS\Doctorado\Data\2016_" 

if not os.path.exists(carpeta_salida):
    os.makedirs(carpeta_salida)

meses = {
    '1. Enero': 1, '2. Febrero': 2, '3. Marzo': 3, '4. Abril': 4, 
    '5. Mayo': 5, '6. Junio': 6, '7. Julio': 7, '8. Agosto': 8, 
    '9. Septiembre': 9, '10. Octubre': 10, '11. Noviembre': 11, '12. Diciembre': 12
}

# Claves de búsqueda con las protecciones de años anteriores
modulos_claves = {
    "caracteristicas_generales": ["caract"],
    "desocupados": ["desocupado", "deocupado", "descupado"], 
    "fuerza_trabajo": ["fuerza"],
    "inactivos": ["inactivos"],
    "ocupados": ["ocupados"], 
    "otras_actividades": ["otras"],
    "otros_ingresos": ["otros ingresos"],
    "hogares": ["vivienda", "hogar"]
}

def quitar_tildes(cadena):
    try:
        s = ''.join(c for c in unicodedata.normalize('NFD', str(cadena)) if unicodedata.category(c) != 'Mn')
        return s.lower()
    except:
        return str(cadena).lower()

def encontrar_archivo_en_zip(lista_archivos_zip, lista_palabras_clave):
    for archivo in lista_archivos_zip:
        archivo_norm = quitar_tildes(archivo.lower())
        if '.dta' in archivo_norm:
            for palabra in lista_palabras_clave:
                palabra_norm = quitar_tildes(palabra)
                if palabra_norm in archivo_norm:
                    # Protección para no mezclar ocupados con desocupados
                    if palabra_norm == "ocupados" and ("desocupado" in archivo_norm or "deocupado" in archivo_norm):
                        continue
                    return archivo
    return None

# =====================================================================
# 2. PROCESAMIENTO
# =====================================================================
diccionario_variables = []

for nombre_modulo_final, palabras_clave in modulos_claves.items():
    print(f"\nProcesando Módulo 2016: {nombre_modulo_final.upper()}")
    datos_modulo = []
    
    for mes_nombre, mes_num in meses.items():
        ruta_zip = os.path.join(ruta_base, f"{mes_nombre}.zip")
        if os.path.exists(ruta_zip):
            with zipfile.ZipFile(ruta_zip, 'r') as z:
                archivo_objetivo = encontrar_archivo_en_zip(z.namelist(), palabras_clave)
                
                if archivo_objetivo:
                    tmp_path = ""
                    try:
                        with tempfile.NamedTemporaryFile(delete=False, suffix=".dta") as tmp:
                            tmp.write(z.read(archivo_objetivo))
                            tmp_path = tmp.name
                        
                        df, meta = pyreadstat.read_dta(tmp_path)
                        df.columns = df.columns.str.lower()
                        df['mes'] = mes_num
                        datos_modulo.append(df)
                        print(f"  [OK] {mes_nombre}")
                    except Exception as e:
                        print(f"  [ERROR] {mes_nombre}: {e}")
                    finally:
                        if tmp_path and os.path.exists(tmp_path): os.remove(tmp_path)
                else:
                    print(f"  [AVISO] No encontrado en {mes_nombre}")

    if datos_modulo:
        df_final = pd.concat(datos_modulo, ignore_index=True)
        # Limpieza para evitar errores de exportación a Stata
        for col in df_final.columns:
            if df_final[col].dtype == 'object':
                df_final[col] = df_final[col].astype(str).replace('nan', '')
        
        df_final.to_stata(os.path.join(carpeta_salida, f"{nombre_modulo_final}.dta"), write_index=False, version=118)
        for var in df_final.columns:
            diccionario_variables.append({"Módulo": nombre_modulo_final, "Variable": var})

# Guardar Diccionario 2016
pd.DataFrame(diccionario_variables).drop_duplicates().to_excel(os.path.join(carpeta_salida, "Diccionario_2016.xlsx"), index=False)
print("\n¡PROCESO 2016 COMPLETADO!")


Procesando Módulo 2016: CARACTERISTICAS_GENERALES
  [OK] 1. Enero
  [OK] 2. Febrero
  [OK] 3. Marzo
  [OK] 4. Abril
  [OK] 5. Mayo
  [OK] 6. Junio
  [OK] 7. Julio
  [OK] 8. Agosto
  [OK] 9. Septiembre
  [OK] 10. Octubre
  [OK] 11. Noviembre
  [OK] 12. Diciembre

Procesando Módulo 2016: DESOCUPADOS
  [OK] 1. Enero
  [OK] 2. Febrero
  [OK] 3. Marzo
  [OK] 4. Abril
  [OK] 5. Mayo
  [OK] 6. Junio
  [OK] 7. Julio
  [OK] 8. Agosto
  [OK] 9. Septiembre
  [OK] 10. Octubre
  [OK] 11. Noviembre
  [OK] 12. Diciembre

Procesando Módulo 2016: FUERZA_TRABAJO
  [OK] 1. Enero
  [OK] 2. Febrero
  [OK] 3. Marzo
  [OK] 4. Abril
  [OK] 5. Mayo
  [OK] 6. Junio
  [OK] 7. Julio
  [OK] 8. Agosto
  [OK] 9. Septiembre
  [OK] 10. Octubre
  [OK] 11. Noviembre
  [OK] 12. Diciembre

Procesando Módulo 2016: INACTIVOS
  [OK] 1. Enero
  [OK] 2. Febrero
  [OK] 3. Marzo
  [OK] 4. Abril
  [OK] 5. Mayo
  [OK] 6. Junio
  [OK] 7. Julio
  [OK] 8. Agosto
  [OK] 9. Septiembre
  [OK] 10. Octubre
  [OK] 11. Noviembre
  [OK] 12.

# 2017


In [7]:
import os
import pandas as pd
import unicodedata
import zipfile
import tempfile
import pyreadstat

# =====================================================================
# 1. CONFIGURACIÓN DE RUTAS (Actualizado para 2017)
# =====================================================================
ruta_base = r"E:\ENTREGA TESIS\Doctorado\Data\2017\GEIH_Empalme_2017" 
carpeta_salida = r"E:\ENTREGA TESIS\Doctorado\Data\2017_" 

if not os.path.exists(carpeta_salida):
    os.makedirs(carpeta_salida)

meses = {
    '1. Enero': 1, '2. Febrero': 2, '3. Marzo': 3, '4. Abril': 4, 
    '5. Mayo': 5, '6. Junio': 6, '7. Julio': 7, '8. Agosto': 8, 
    '9. Septiembre': 9, '10. Octubre': 10, '11. Noviembre': 11, '12. Diciembre': 12
}

# Claves de búsqueda estables
modulos_claves = {
    "caracteristicas_generales": ["caract"],
    "desocupados": ["desocupado", "deocupado", "descupado"], 
    "fuerza_trabajo": ["fuerza"],
    "inactivos": ["inactivos"],
    "ocupados": ["ocupados"], 
    "otras_actividades": ["otras"],
    "otros_ingresos": ["otros ingresos"],
    "hogares": ["vivienda", "hogar"]
}

def quitar_tildes(cadena):
    try:
        s = ''.join(c for c in unicodedata.normalize('NFD', str(cadena)) if unicodedata.category(c) != 'Mn')
        return s.lower()
    except:
        return str(cadena).lower()

def encontrar_archivo_en_zip(lista_archivos_zip, lista_palabras_clave):
    for archivo in lista_archivos_zip:
        archivo_norm = quitar_tildes(archivo.lower())
        if '.dta' in archivo_norm:
            for palabra in lista_palabras_clave:
                palabra_norm = quitar_tildes(palabra)
                if palabra_norm in archivo_norm:
                    # Evitar colisión entre ocupados y desocupados
                    if palabra_norm == "ocupados" and ("desocupado" in archivo_norm or "deocupado" in archivo_norm):
                        continue
                    return archivo
    return None

# =====================================================================
# 2. PROCESAMIENTO
# =====================================================================
diccionario_variables = []

for nombre_modulo_final, palabras_clave in modulos_claves.items():
    print(f"\nProcesando Módulo 2017: {nombre_modulo_final.upper()}")
    datos_modulo = []
    
    for mes_nombre, mes_num in meses.items():
        ruta_zip = os.path.join(ruta_base, f"{mes_nombre}.zip")
        if os.path.exists(ruta_zip):
            with zipfile.ZipFile(ruta_zip, 'r') as z:
                archivo_objetivo = encontrar_archivo_en_zip(z.namelist(), palabras_clave)
                
                if archivo_objetivo:
                    tmp_path = ""
                    try:
                        with tempfile.NamedTemporaryFile(delete=False, suffix=".dta") as tmp:
                            tmp.write(z.read(archivo_objetivo))
                            tmp_path = tmp.name
                        
                        # pyreadstat es ideal para estos archivos más pesados
                        df, meta = pyreadstat.read_dta(tmp_path)
                        df.columns = df.columns.str.lower()
                        df['mes'] = mes_num
                        datos_modulo.append(df)
                        print(f"  [OK] {mes_nombre}")
                    except Exception as e:
                        print(f"  [ERROR] {mes_nombre}: {e}")
                    finally:
                        if tmp_path and os.path.exists(tmp_path): os.remove(tmp_path)
                else:
                    print(f"  [AVISO] No encontrado en {mes_nombre}")

    if datos_modulo:
        df_final = pd.concat(datos_modulo, ignore_index=True)
        # Sanitizar strings para Stata
        for col in df_final.columns:
            if df_final[col].dtype == 'object':
                df_final[col] = df_final[col].astype(str).replace('nan', '')
        
        df_final.to_stata(os.path.join(carpeta_salida, f"{nombre_modulo_final}.dta"), write_index=False, version=118)
        for var in df_final.columns:
            diccionario_variables.append({"Módulo": nombre_modulo_final, "Variable": var})

# Guardar Diccionario 2017
pd.DataFrame(diccionario_variables).drop_duplicates().to_excel(os.path.join(carpeta_salida, "Diccionario_2017.xlsx"), index=False)
print("\n¡PROCESO 2017 COMPLETADO!")


Procesando Módulo 2017: CARACTERISTICAS_GENERALES
  [OK] 1. Enero
  [OK] 2. Febrero
  [OK] 3. Marzo
  [OK] 4. Abril
  [OK] 5. Mayo
  [OK] 6. Junio
  [OK] 7. Julio
  [OK] 8. Agosto
  [OK] 9. Septiembre
  [OK] 10. Octubre
  [OK] 11. Noviembre
  [OK] 12. Diciembre

Procesando Módulo 2017: DESOCUPADOS
  [OK] 1. Enero
  [OK] 2. Febrero
  [OK] 3. Marzo
  [OK] 4. Abril
  [OK] 5. Mayo
  [OK] 6. Junio
  [OK] 7. Julio
  [OK] 8. Agosto
  [OK] 9. Septiembre
  [OK] 10. Octubre
  [OK] 11. Noviembre
  [OK] 12. Diciembre

Procesando Módulo 2017: FUERZA_TRABAJO
  [OK] 1. Enero
  [OK] 2. Febrero
  [OK] 3. Marzo
  [OK] 4. Abril
  [OK] 5. Mayo
  [OK] 6. Junio
  [OK] 7. Julio
  [OK] 8. Agosto
  [OK] 9. Septiembre
  [OK] 10. Octubre
  [OK] 11. Noviembre
  [OK] 12. Diciembre

Procesando Módulo 2017: INACTIVOS
  [OK] 1. Enero
  [OK] 2. Febrero
  [OK] 3. Marzo
  [OK] 4. Abril
  [OK] 5. Mayo
  [OK] 6. Junio
  [OK] 7. Julio
  [OK] 8. Agosto
  [OK] 9. Septiembre
  [OK] 10. Octubre
  [OK] 11. Noviembre
  [OK] 12.

# 2018

In [10]:
import os
import pandas as pd
import unicodedata
import zipfile
import tempfile
import pyreadstat

# =====================================================================
# 1. CONFIGURACIÓN DE RUTAS (Actualizado para 2018)
# =====================================================================
ruta_base = r"E:\ENTREGA TESIS\Doctorado\Data\2018\GEIH_Empalme_2018" 
carpeta_salida = r"E:\ENTREGA TESIS\Doctorado\Data\2018_" 

if not os.path.exists(carpeta_salida):
    os.makedirs(carpeta_salida)

meses = {
    '1. Enero': 1, '2. Febrero': 2, '3. Marzo': 3, '4. Abril': 4, 
    '5. Mayo': 5, '6. Junio': 6, '7. Julio': 7, '8. Agosto': 8, 
    '9. Septiembre': 9, '10. Octubre': 10, '11. Noviembre': 11, '12. Diciembre': 12
}

# Claves de búsqueda robustas
modulos_claves = {
    "caracteristicas_generales": ["caract"],
    "desocupados": ["desocupado", "deocupado", "descupado"], 
    "fuerza_trabajo": ["fuerza"],
    "inactivos": ["inactivos"],
    "ocupados": ["ocupados"], 
    "otras_actividades": ["otras"],
    "otros_ingresos": ["otros ingresos", "tros ingresos"],
    "hogares": ["vivienda", "hogar"]
}

def quitar_tildes(cadena):
    try:
        s = ''.join(c for c in unicodedata.normalize('NFD', str(cadena)) if unicodedata.category(c) != 'Mn')
        return s.lower()
    except:
        return str(cadena).lower()

def encontrar_archivo_en_zip(lista_archivos_zip, lista_palabras_clave):
    for archivo in lista_archivos_zip:
        archivo_norm = quitar_tildes(archivo.lower())
        if '.dta' in archivo_norm:
            for palabra in lista_palabras_clave:
                palabra_norm = quitar_tildes(palabra)
                if palabra_norm in archivo_norm:
                    # Protección para no mezclar ocupados con desocupados
                    if palabra_norm == "ocupados" and ("desocupado" in archivo_norm or "deocupado" in archivo_norm):
                        continue
                    return archivo
    return None

# =====================================================================
# 2. PROCESAMIENTO
# =====================================================================
diccionario_variables = []

for nombre_modulo_final, palabras_clave in modulos_claves.items():
    print(f"\nProcesando Módulo 2018: {nombre_modulo_final.upper()}")
    datos_modulo = []
    
    for mes_nombre, mes_num in meses.items():
        ruta_zip = os.path.join(ruta_base, f"{mes_nombre}.zip")
        if os.path.exists(ruta_zip):
            with zipfile.ZipFile(ruta_zip, 'r') as z:
                archivo_objetivo = encontrar_archivo_en_zip(z.namelist(), palabras_clave)
                
                if archivo_objetivo:
                    tmp_path = ""
                    try:
                        with tempfile.NamedTemporaryFile(delete=False, suffix=".dta") as tmp:
                            tmp.write(z.read(archivo_objetivo))
                            tmp_path = tmp.name
                        
                        df, meta = pyreadstat.read_dta(tmp_path)
                        df.columns = df.columns.str.lower()
                        df['mes'] = mes_num
                        datos_modulo.append(df)
                        print(f"  [OK] {mes_nombre}")
                    except Exception as e:
                        print(f"  [ERROR] {mes_nombre}: {e}")
                    finally:
                        if tmp_path and os.path.exists(tmp_path): os.remove(tmp_path)
                else:
                    print(f"  [AVISO] No encontrado en {mes_nombre}")

    if datos_modulo:
        df_final = pd.concat(datos_modulo, ignore_index=True)
        # Limpieza para Stata
        for col in df_final.columns:
            if df_final[col].dtype == 'object':
                df_final[col] = df_final[col].astype(str).replace('nan', '')
        
        df_final.to_stata(os.path.join(carpeta_salida, f"{nombre_modulo_final}.dta"), write_index=False, version=118)
        for var in df_final.columns:
            diccionario_variables.append({"Módulo": nombre_modulo_final, "Variable": var})

# Guardar Diccionario 2018
pd.DataFrame(diccionario_variables).drop_duplicates().to_excel(os.path.join(carpeta_salida, "Diccionario_2018.xlsx"), index=False)
print("\n¡PROCESO 2018 COMPLETADO!")


Procesando Módulo 2018: CARACTERISTICAS_GENERALES
  [OK] 1. Enero
  [OK] 2. Febrero
  [OK] 3. Marzo
  [OK] 4. Abril
  [OK] 5. Mayo
  [OK] 6. Junio
  [OK] 7. Julio
  [OK] 8. Agosto
  [OK] 9. Septiembre
  [OK] 10. Octubre
  [OK] 11. Noviembre
  [OK] 12. Diciembre

Procesando Módulo 2018: DESOCUPADOS
  [OK] 1. Enero
  [OK] 2. Febrero
  [OK] 3. Marzo
  [OK] 4. Abril
  [OK] 5. Mayo
  [OK] 6. Junio
  [OK] 7. Julio
  [OK] 8. Agosto
  [OK] 9. Septiembre
  [OK] 10. Octubre
  [OK] 11. Noviembre
  [OK] 12. Diciembre

Procesando Módulo 2018: FUERZA_TRABAJO
  [OK] 1. Enero
  [OK] 2. Febrero
  [OK] 3. Marzo
  [OK] 4. Abril
  [OK] 5. Mayo
  [OK] 6. Junio
  [OK] 7. Julio
  [OK] 8. Agosto
  [OK] 9. Septiembre
  [OK] 10. Octubre
  [OK] 11. Noviembre
  [OK] 12. Diciembre

Procesando Módulo 2018: INACTIVOS
  [OK] 1. Enero
  [OK] 2. Febrero
  [OK] 3. Marzo
  [OK] 4. Abril
  [OK] 5. Mayo
  [OK] 6. Junio
  [OK] 7. Julio
  [OK] 8. Agosto
  [OK] 9. Septiembre
  [OK] 10. Octubre
  [OK] 11. Noviembre
  [OK] 12.

# 2019

In [9]:
import os
import pandas as pd
import unicodedata
import zipfile
import tempfile
import pyreadstat

# =====================================================================
# 1. CONFIGURACIÓN DE RUTAS (Actualizado para 2019)
# =====================================================================
# Ruta basada en tu mensaje: E:\ENTREGA TESIS\Doctorado\Data\2019
ruta_base = r"E:\ENTREGA TESIS\Doctorado\Data\2019" 
carpeta_salida = r"E:\ENTREGA TESIS\Doctorado\Data\2019_" 

if not os.path.exists(carpeta_salida):
    os.makedirs(carpeta_salida)

meses = {
    '1. Enero': 1, '2. Febrero': 2, '3. Marzo': 3, '4. Abril': 4, 
    '5. Mayo': 5, '6. Junio': 6, '7. Julio': 7, '8. Agosto': 8, 
    '9. Septiembre': 9, '10. Octubre': 10, '11. Noviembre': 11, '12. Diciembre': 12
}

# Claves de búsqueda consistentes
modulos_claves = {
    "caracteristicas_generales": ["caract"],
    "desocupados": ["desocupado", "deocupado", "descupado"], 
    "fuerza_trabajo": ["fuerza"],
    "inactivos": ["inactivos"],
    "ocupados": ["ocupados"], 
    "otras_actividades": ["otras"],
    "otros_ingresos": ["otros ingresos"],
    "hogares": ["vivienda", "hogar"]
}

def quitar_tildes(cadena):
    try:
        s = ''.join(c for c in unicodedata.normalize('NFD', str(cadena)) if unicodedata.category(c) != 'Mn')
        return s.lower()
    except:
        return str(cadena).lower()

def encontrar_archivo_en_zip(lista_archivos_zip, lista_palabras_clave):
    for archivo in lista_archivos_zip:
        archivo_norm = quitar_tildes(archivo.lower())
        if '.dta' in archivo_norm:
            for palabra in lista_palabras_clave:
                palabra_norm = quitar_tildes(palabra)
                if palabra_norm in archivo_norm:
                    # Protección para evitar colisión Ocupados vs Desocupados
                    if palabra_norm == "ocupados" and ("desocupado" in archivo_norm or "deocupado" in archivo_norm):
                        continue
                    return archivo
    return None

# =====================================================================
# 2. PROCESAMIENTO
# =====================================================================
diccionario_variables = []

for nombre_modulo_final, palabras_clave in modulos_claves.items():
    print(f"\nProcesando Módulo 2019: {nombre_modulo_final.upper()}")
    datos_modulo = []
    
    for mes_nombre, mes_num in meses.items():
        ruta_zip = os.path.join(ruta_base, f"{mes_nombre}.zip")
        if os.path.exists(ruta_zip):
            with zipfile.ZipFile(ruta_zip, 'r') as z:
                archivo_objetivo = encontrar_archivo_en_zip(z.namelist(), palabras_clave)
                
                if archivo_objetivo:
                    tmp_path = ""
                    try:
                        with tempfile.NamedTemporaryFile(delete=False, suffix=".dta") as tmp:
                            tmp.write(z.read(archivo_objetivo))
                            tmp_path = tmp.name
                        
                        df, meta = pyreadstat.read_dta(tmp_path)
                        df.columns = df.columns.str.lower()
                        df['mes'] = mes_num
                        datos_modulo.append(df)
                        print(f"  [OK] {mes_nombre}")
                    except Exception as e:
                        print(f"  [ERROR] {mes_nombre}: {e}")
                    finally:
                        if tmp_path and os.path.exists(tmp_path): os.remove(tmp_path)
                else:
                    print(f"  [AVISO] No encontrado en {mes_nombre}")

    if datos_modulo:
        df_final = pd.concat(datos_modulo, ignore_index=True)
        # Limpieza para Stata
        for col in df_final.columns:
            if df_final[col].dtype == 'object':
                df_final[col] = df_final[col].astype(str).replace('nan', '')
        
        df_final.to_stata(os.path.join(carpeta_salida, f"{nombre_modulo_final}.dta"), write_index=False, version=118)
        for var in df_final.columns:
            diccionario_variables.append({"Módulo": nombre_modulo_final, "Variable": var})

# Guardar Diccionario 2019
pd.DataFrame(diccionario_variables).drop_duplicates().to_excel(os.path.join(carpeta_salida, "Diccionario_2019.xlsx"), index=False)
print("\n¡PROCESO 2019 COMPLETADO!")


Procesando Módulo 2019: CARACTERISTICAS_GENERALES
  [OK] 1. Enero
  [OK] 2. Febrero
  [OK] 3. Marzo
  [OK] 4. Abril
  [OK] 5. Mayo
  [OK] 6. Junio
  [OK] 7. Julio
  [OK] 8. Agosto
  [OK] 9. Septiembre
  [OK] 10. Octubre
  [OK] 11. Noviembre
  [OK] 12. Diciembre

Procesando Módulo 2019: DESOCUPADOS
  [OK] 1. Enero
  [OK] 2. Febrero
  [OK] 3. Marzo
  [OK] 4. Abril
  [OK] 5. Mayo
  [OK] 6. Junio
  [OK] 7. Julio
  [OK] 8. Agosto
  [OK] 9. Septiembre
  [OK] 10. Octubre
  [OK] 11. Noviembre
  [OK] 12. Diciembre

Procesando Módulo 2019: FUERZA_TRABAJO
  [OK] 1. Enero
  [OK] 2. Febrero
  [OK] 3. Marzo
  [OK] 4. Abril
  [OK] 5. Mayo
  [OK] 6. Junio
  [OK] 7. Julio
  [OK] 8. Agosto
  [OK] 9. Septiembre
  [OK] 10. Octubre
  [OK] 11. Noviembre
  [OK] 12. Diciembre

Procesando Módulo 2019: INACTIVOS
  [OK] 1. Enero
  [OK] 2. Febrero
  [OK] 3. Marzo
  [OK] 4. Abril
  [OK] 5. Mayo
  [OK] 6. Junio
  [OK] 7. Julio
  [OK] 8. Agosto
  [OK] 9. Septiembre
  [OK] 10. Octubre
  [OK] 11. Noviembre
  [OK] 12.

# contabilizacion de numero de observaciones para confirmación 

In [3]:
import os
import pandas as pd
import pyreadstat

# =====================================================================
# 1. CONFIGURACIÓN DE LA RUTA RAÍZ
# =====================================================================
# Esta es la carpeta padre donde viven tus carpetas "2007_", "2008_", etc.
ruta_principal = r"E:\ENTREGA TESIS\Doctorado\Data"

anios = list(range(2007, 2020))

modulos = [
    "caracteristicas_generales",
    "desocupados",
    "fuerza_trabajo",
    "inactivos",
    "ocupados",
    "otras_actividades",
    "otros_ingresos",
    "hogares"
]

resultados = []

print("Iniciando el escaneo supersónico de observaciones por año y módulo...\n")

# =====================================================================
# 2. BUCLE DE LECTURA DE METADATOS
# =====================================================================
for anio in anios:
    carpeta_anio = os.path.join(ruta_principal, f"{anio}_")
    
    if not os.path.exists(carpeta_anio):
        print(f"  [AVISO] No se encontró la carpeta: {carpeta_anio}")
        continue
        
    print(f"Contando observaciones del año {anio}...")
    
    for modulo in modulos:
        ruta_dta = os.path.join(carpeta_anio, f"{modulo}.dta")
        
        if os.path.exists(ruta_dta):
            try:
                # TRUCO MÁGICO: metadataonly=True lee el conteo sin cargar los microdatos a la RAM
                df_dummy, meta = pyreadstat.read_dta(ruta_dta, metadataonly=True)
                num_obs = meta.number_rows
                
                resultados.append({
                    "Año": anio,
                    "Módulo": modulo,
                    "Observaciones": num_obs
                })
            except Exception as e:
                print(f"  [ERROR] Al leer {ruta_dta}: {e}")
                resultados.append({
                    "Año": anio,
                    "Módulo": modulo,
                    "Observaciones": None
                })
        else:
            # Si en algún año no existe el módulo, le ponemos 0
            resultados.append({
                "Año": anio,
                "Módulo": modulo,
                "Observaciones": 0
            })

# =====================================================================
# 3. EXPORTACIÓN A TABLA EXCEL RESUMEN
# =====================================================================
if resultados:
    df_resultados = pd.DataFrame(resultados)
    
    # Pivoteamos la tabla: Años en las filas, Módulos en las columnas
    df_pivot = df_resultados.pivot(index="Año", columns="Módulo", values="Observaciones")
    
    # Ordenamos las columnas para mayor claridad
    df_pivot = df_pivot[modulos]
    
    # Formateamos los números con separador de miles para que se vean elegantes
    df_formateado = df_pivot.applymap(lambda x: f"{int(x):,}".replace(",", ".") if pd.notnull(x) else "Error")
    
    ruta_excel = os.path.join(ruta_principal, "Tabla_Observaciones_GEIH_2007_2019.xlsx")
    df_formateado.to_excel(ruta_excel)
    
    print("\n" + "="*60)
    print("¡CONTEO COMPLETADO CON ÉXITO!")
    print(f"Tu tabla resumen se guardó en: {ruta_excel}")
    print("="*60 + "\n")
    print("Vista previa de tu Anexo Metodológico:")
    print(df_formateado.head())
else:
    print("\nNo se encontraron datos para procesar. Revisa las rutas.")

Iniciando el escaneo supersónico de observaciones por año y módulo...

Contando observaciones del año 2007...
Contando observaciones del año 2008...
Contando observaciones del año 2009...
Contando observaciones del año 2010...
Contando observaciones del año 2011...
Contando observaciones del año 2012...
Contando observaciones del año 2013...
Contando observaciones del año 2014...
Contando observaciones del año 2015...
Contando observaciones del año 2016...
Contando observaciones del año 2017...
Contando observaciones del año 2018...
Contando observaciones del año 2019...


C:\Users\USUARIO\AppData\Local\Temp\ipykernel_11488\34910543.py:82: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df_formateado = df_pivot.applymap(lambda x: f"{int(x):,}".replace(",", ".") if pd.notnull(x) else "Error")



¡CONTEO COMPLETADO CON ÉXITO!
Tu tabla resumen se guardó en: E:\ENTREGA TESIS\Doctorado\Data\Tabla_Observaciones_GEIH_2007_2019.xlsx

Vista previa de tu Anexo Metodológico:
Módulo caracteristicas_generales desocupados fuerza_trabajo inactivos  \
Año                                                                     
2007                     838.421      49.515        653.136   274.135   
2008                     823.814      49.492        646.651   269.490   
2009                     816.242      52.252        636.254   251.410   
2010                     842.221      53.532        607.922   203.185   
2011                     847.216      51.587        617.822   202.284   

Módulo ocupados otras_actividades otros_ingresos  hogares  
Año                                                        
2007    329.486           653.136        653.132  219.978  
2008    327.669           646.649        646.639  221.585  
2009    332.592           636.253        636.250  220.989  
2010    351.20

Revisión de duplicados 

In [4]:
import pandas as pd
import os

# Ruta a tu archivo de 2009 que tiene el problema de doble conteo
ruta_dta = r"E:\ENTREGA TESIS\Doctorado\Data\2009_\caracteristicas_generales.dta"

print("Cargando datos para comprobación...")
# Solo cargamos las columnas necesarias para no saturar la RAM
df = pd.read_stata(ruta_dta, columns=['directorio', 'secuencia_p', 'orden', 'zona_area'])

# Separamos las bases según la zona que le asignó nuestro script
df_area = df[df['zona_area'] == 1].copy()
df_cabecera = df[df['zona_area'] == 2].copy()

print(f"\nObservaciones en carpeta 'Área': {len(df_area):,}")
print(f"Observaciones en carpeta 'Cabecera': {len(df_cabecera):,}")

# Creamos un ID único pegando los tres identificadores
df_area['ID_UNICO'] = df_area['directorio'].astype(str) + "_" + df_area['secuencia_p'].astype(str) + "_" + df_area['orden'].astype(str)
df_cabecera['ID_UNICO'] = df_cabecera['directorio'].astype(str) + "_" + df_cabecera['secuencia_p'].astype(str) + "_" + df_cabecera['orden'].astype(str)

# Buscamos cuántas personas de 'Área' existen idénticas dentro de 'Cabecera'
duplicados = df_area['ID_UNICO'].isin(df_cabecera['ID_UNICO']).sum()
porcentaje = (duplicados / len(df_area)) * 100

print("\n--- RESULTADO DE LA AUDITORÍA ---")
print(f"Personas de 'Área' encontradas DENTRO de 'Cabecera': {duplicados:,}")
print(f"Porcentaje de solapamiento: {porcentaje:.2f}%")

Cargando datos para comprobación...

Observaciones en carpeta 'Área': 0
Observaciones en carpeta 'Cabecera': 731,025

--- RESULTADO DE LA AUDITORÍA ---
Personas de 'Área' encontradas DENTRO de 'Cabecera': 0
Porcentaje de solapamiento: nan%


C:\Users\USUARIO\AppData\Local\Temp\ipykernel_11488\1387048228.py:24: RuntimeWarning: invalid value encountered in scalar divide
  porcentaje = (duplicados / len(df_area)) * 100


revision numero de identificadores unicos de hogar en caracteristicas generales de personas debe ser igual al modulo hogar 

In [7]:
import os
import pandas as pd
import pyreadstat

# =====================================================================
# CONFIGURACIÓN DE RUTAS
# =====================================================================
ruta_principal = r"E:\ENTREGA TESIS\Doctorado\Data"
anios = list(range(2007, 2020))
verificaciones = []

print("Iniciando auditoría corregida: Vivienda (directorio) + Hogar (secuencia_p)...\n")

for anio in anios:
    carpeta_anio = os.path.join(ruta_principal, f"{anio}_")
    archivo_personas = os.path.join(carpeta_anio, "caracteristicas_generales.dta")
    archivo_hogares = os.path.join(carpeta_anio, "hogares.dta")
    
    if os.path.exists(archivo_personas) and os.path.exists(archivo_hogares):
        try:
            # 1. Cargar llaves del módulo de Hogares (Llave: directorio + secuencia_p)
            df_h, _ = pyreadstat.read_dta(archivo_hogares, usecols=['directorio', 'secuencia_p'])
            hogares_en_modulo_hogares = len(df_h.drop_duplicates(subset=['directorio', 'secuencia_p']))
            
            # 2. Cargar llaves del módulo de Personas (Llave: directorio + secuencia_p)
            df_p, _ = pyreadstat.read_dta(archivo_personas, usecols=['directorio', 'secuencia_p'])
            hogares_unicos_en_personas = len(df_p.drop_duplicates(subset=['directorio', 'secuencia_p']))
            
            # 3. Comparación
            diferencia = hogares_unicos_en_personas - hogares_en_modulo_hogares
            estado = "OK" if abs(diferencia) < 5 else "DISCREPANCIA" # Margen mínimo por limpieza DANE
            
            verificaciones.append({
                "Año": anio,
                "Total Hogares (Módulo Hogares)": hogares_en_modulo_hogares,
                "Hogares Únicos (en Personas)": hogares_unicos_en_personas,
                "Diferencia": diferencia,
                "Estado": estado
            })
            print(f"  [OK] Año {anio} procesado.")
            
        except Exception as e:
            print(f"  [ERROR] En el año {anio}: {e}")
    else:
        print(f"  [AVISO] Archivos no encontrados en {anio}_")

# =====================================================================
# REPORTE FINAL
# =====================================================================
df_reporte = pd.DataFrame(verificaciones)

# Formateo numérico
columnas_num = ["Total Hogares (Módulo Hogares)", "Hogares Únicos (en Personas)", "Diferencia"]
df_reporte[columnas_num] = df_reporte[columnas_num].applymap(lambda x: f"{int(x):,}".replace(",", "."))

print("\n" + "="*95)
print("AUDITORÍA DE INTEGRIDAD: IDENTIFICACIÓN CORRECTA DE HOGARES (directorio + secuencia_p)")
print("="*95)
print(df_reporte.to_string(index=False))
print("="*95)

ruta_excel = os.path.join(ruta_principal, "Auditoria_Final_Hogares_GEIH.xlsx")
df_reporte.to_excel(ruta_excel, index=False)

Iniciando auditoría corregida: Vivienda (directorio) + Hogar (secuencia_p)...

  [OK] Año 2007 procesado.
  [OK] Año 2008 procesado.
  [OK] Año 2009 procesado.
  [OK] Año 2010 procesado.
  [OK] Año 2011 procesado.
  [OK] Año 2012 procesado.
  [OK] Año 2013 procesado.
  [OK] Año 2014 procesado.
  [OK] Año 2015 procesado.
  [OK] Año 2016 procesado.
  [OK] Año 2017 procesado.
  [OK] Año 2018 procesado.
  [OK] Año 2019 procesado.

AUDITORÍA DE INTEGRIDAD: IDENTIFICACIÓN CORRECTA DE HOGARES (directorio + secuencia_p)
 Año Total Hogares (Módulo Hogares) Hogares Únicos (en Personas) Diferencia       Estado
2007                        219.978                      221.048      1.070 DISCREPANCIA
2008                        221.585                      221.988        403 DISCREPANCIA
2009                        220.916                      221.108        192 DISCREPANCIA
2010                        232.972                      232.975          3           OK
2011                        237.061  

C:\Users\USUARIO\AppData\Local\Temp\ipykernel_11488\2426412855.py:54: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df_reporte[columnas_num] = df_reporte[columnas_num].applymap(lambda x: f"{int(x):,}".replace(",", "."))


Creacion de diccionario de variables

In [ ]:
import os
import pandas as pd
import pyreadstat

# =====================================================================
# 1. CONFIGURACIÓN
# =====================================================================
ruta_principal = r"E:\ENTREGA TESIS\Doctorado\Data"
anios = list(range(2007, 2020))

# Vamos a enfocarnos en los módulos vitales para tu panel econométrico
# (Puedes agregar 'hogares' o 'otros_ingresos' si los necesitas)
modulos = [
    "caracteristicas_generales",
    "fuerza_trabajo",
    "ocupados",
    "desocupados",
    "inactivos",
    "hogares",
    "otras_actividades", 
    "otros_ingresos"
]

registros = []

print("Extrayendo variables y descripciones (Metadatos) de 2007 a 2019...\n")

# =====================================================================
# 2. EXTRACCIÓN DE METADATOS
# =====================================================================
for anio in anios:
    carpeta_anio = os.path.join(ruta_principal, f"{anio}_")
    
    for modulo in modulos:
        ruta_dta = os.path.join(carpeta_anio, f"{modulo}.dta")
        
        if os.path.exists(ruta_dta):
            try:
                # metadataonly=True lee las descripciones sin cargar la base pesada
                _, meta = pyreadstat.read_dta(ruta_dta, metadataonly=True)
                
                diccionario_etiquetas = meta.column_names_to_labels
                
                for var_nombre in meta.column_names:
                    # Extraer la descripción (si no tiene, se deja en blanco)
                    descripcion = diccionario_etiquetas.get(var_nombre, "")
                    
                    registros.append({
                        "Modulo": modulo,
                        "Variable_Original": var_nombre,
                        "Descripcion_Oficial": descripcion,
                        "Año": anio,
                        "Presencia": 1
                    })
            except Exception as e:
                print(f"  [ERROR] No se pudo leer metadatos de {modulo} en {anio}: {e}")

# =====================================================================
# 3. CREACIÓN DE LA MATRIZ PIVOTE (PIEDRA ROSETTA)
# =====================================================================
df_crudo = pd.DataFrame(registros)

if not df_crudo.empty:
    # 1. Agrupar para obtener la descripción más común de cada variable
    # (A veces el DANE cambia ligeramente la redacción de un año a otro)
    df_descripciones = df_crudo.groupby(['Modulo', 'Variable_Original'])['Descripcion_Oficial'].first().reset_index()
    
    # 2. Crear la matriz de presencia (Años en las columnas)
    df_presencia = df_crudo.pivot_table(
        index=['Modulo', 'Variable_Original'], 
        columns='Año', 
        values='Presencia', 
        fill_value=0
    ).reset_index()
    
    # 3. Unir la descripción con la matriz
    df_rosetta = pd.merge(df_presencia, df_descripciones, on=['Modulo', 'Variable_Original'])
    
    # 4. Ordenar columnas y añadir la columna para tu trabajo manual
    cols_ordenadas = ['Modulo', 'Variable_Original', 'Descripcion_Oficial'] + anios
    df_rosetta = df_rosetta[cols_ordenadas]
    
    # ¡LA COLUMNA MÁGICA!
    df_rosetta['Variable_Homologada'] = "" 
    
    # =====================================================================
    # 4. GUARDAR EXCEL
    # =====================================================================
    ruta_excel = os.path.join(ruta_principal, "Piedra_Rosetta_GEIH_2007_2019.xlsx")
    df_rosetta.to_excel(ruta_excel, index=False)
    
    print("\n" + "="*80)
    print("¡PIEDRA ROSETTA GENERADA CON ÉXITO!")
    print(f"Ruta: {ruta_excel}")
    print("="*80)
    print("\nSiguiente paso: Abre el Excel y llena la columna 'Variable_Homologada'.")
else:
    print("No se encontraron datos.")

Extrayendo variables y descripciones (Metadatos) de 2007 a 2019...


¡PIEDRA ROSETTA GENERADA CON ÉXITO!
Ruta: E:\ENTREGA TESIS\Doctorado\Data\Piedra_Rosetta_GEIH_2007_2019v1.xlsx

Siguiente paso: Abre el Excel y llena la columna 'Variable_Homologada'.


Descartare el 2007 voy a empezar a pegar 2008 para ver como me va: 

In [3]:
import os
import pandas as pd
import pyreadstat

# =====================================================================
# 1. CONFIGURACIÓN
# =====================================================================
ruta_2008 = r"E:\ENTREGA TESIS\Doctorado\Data\2008_"
ruta_salida = r"E:\ENTREGA TESIS\Doctorado\Data\Panel_2008_Preliminar.dta"

# =====================================================================
# 2. DEFINICIÓN DE VARIABLES (Solo las que tienen '1' en 2008)
# =====================================================================
vars_cg = ['directorio', 'secuencia_p', 'orden', 'area', 'clase', 'dpto', 'esc', 'fex_c_2011', 'mes', 'p6020', 'p6040', 'p6050', 'p6070', 'p6100', 'p6110', 'p6170', 'p6210', 'p6210s1', 'p6220']
vars_hog = ['directorio', 'secuencia_p', 'p4030s1a1']
vars_ocu = ['directorio', 'secuencia_p', 'orden', 'inglabo', 'oficio', 'p6430', 'p6450', 'p6585s4', 'p6765', 'p6772', 'p6773', 'p6773s1', 'p6870', 'p6920', 'p6930', 'p6940', 'rama2d', 'rama4d']
vars_des = ['directorio', 'secuencia_p', 'orden', 'dsi', 'rama2d_d', 'rama4d_d']
vars_oact = ['directorio', 'secuencia_p', 'orden', 'p7480s8']
vars_oing = ['directorio', 'secuencia_p', 'orden', 'p7500s1a1', 'p7500s2a1', 'p7500s3a1', 'p7510s1a1', 'p7510s2a1', 'p7510s3a1', 'p7510s5a1', 'p7510s6a1', 'p7510s7a1']

# Diccionario de Etiquetas (Añadimos 'ano')
etiquetas_stata = {
    'ano': 'año', # <--- Etiqueta para nuestra nueva variable
    'area': 'area', 'clase': 'clase', 'directorio': 'directorio', 'dpto': 'dpto', 
    'esc': 'escolaridad', 'fex_c_2011': 'factor', 'mes': 'mes', 'orden': 'orden', 
    'p6020': 'sexo', 'p6040': 'edad', 'p6050': 'jefe_hogar', 'p6070': 'estado_civil', 
    'p6100': 'regimen_salud', 'p6110': 'quien_paga_regimen_salud', 'p6170': 'asistencia_escuela', 
    'p6210': 'nivel_edu', 'p6210s1': 'año_aprobado', 'p6220': 'diploma', 'secuencia_p': 'secuencia_p',
    'p4030s1a1': 'estrato', 'inglabo': 'ingreso_laboral', 'oficio': 'oficio', 
    'p6430': 'posicion_ocupacional', 'p6450': 'tipo_contrato', 'p6585s4': 'subsidio_edu', 
    'p6765': 'formas_trabajo', 'p6772': 'registro_merca', 'p6773': 'renovacion', 
    'p6773s1': 'año renova', 'p6870': 'personas_empresa', 'p6920': 'pension', 
    'p6930': 'tipo_fondo', 'p6940': 'quien_paga_regimen_pension', 
    'rama2d': 'rama2d', 'rama4d': 'rama4d', 'dsi': 'desocupados', 'rama2d_d': 'rama2d_d', 
    'rama4d_d': 'rama4d_d', 'p7480s8': 'cursos_cap', 'p7500s1a1': 'ing_arriendos', 
    'p7500s2a1': 'ing_pensiones', 'p7500s3a1': 'ing_aliment', 'p7510s1a1': 'ing_residentes', 
    'p7510s2a1': 'ing_noreside', 'p7510s3a1': 'ing_gob', 'p7510s5a1': 'ing_ahorr', 
    'p7510s6a1': 'ing_cesan', 'p7510s7a1': 'ing_otros'
}

# =====================================================================
# 3. FUNCIÓN DE LECTURA ROBUSTA
# =====================================================================
def cargar_y_filtrar(nombre_archivo, vars_a_mantener):
    ruta_completa = os.path.join(ruta_2008, nombre_archivo)
    if not os.path.exists(ruta_completa):
        print(f"  [AVISO] No se encontró el módulo: {nombre_archivo}")
        return None
    
    # Leer el DTA
    df, meta = pyreadstat.read_dta(ruta_completa)
    
    # Convertir nombres de columnas a minúsculas por seguridad
    df.columns = [c.lower() for c in df.columns]
    
    # Conservar solo las variables que realmente existen en el DataFrame
    cols_finales = [c for c in vars_a_mantener if c in df.columns]
    
    print(f"  [OK] {nombre_archivo} cargado con {len(cols_finales)} variables clave.")
    return df[cols_finales]

# =====================================================================
# 4. PROCESO DE MERGE (PEGADO)
# =====================================================================
print("Iniciando construcción de base 2008...\n")

# A. Base Maestra: Características Generales
df_master = cargar_y_filtrar("caracteristicas_generales.dta", vars_cg)

# B. Pegar Hogares (Llaves: directorio, secuencia_p)
df_hogares = cargar_y_filtrar("hogares.dta", vars_hog)
if df_hogares is not None:
    df_master = pd.merge(df_master, df_hogares, on=['directorio', 'secuencia_p'], how='left')

# C. Pegar Módulos de Personas (Llaves: directorio, secuencia_p, orden)
modulos_extra = [
    ("ocupados.dta", vars_ocu),
    ("desocupados.dta", vars_des),
    ("otras_actividades.dta", vars_oact),
    ("otros_ingresos.dta", vars_oing)
]

for archivo, variables in modulos_extra:
    df_temp = cargar_y_filtrar(archivo, variables)
    if df_temp is not None:
        df_master = pd.merge(df_master, df_temp, on=['directorio', 'secuencia_p', 'orden'], how='left')

# =====================================================================
# 5. CREACIÓN DE VARIABLES NUEVAS Y EXPORTACIÓN A STATA
# =====================================================================

# Crear la variable de Año
print("\nCreando la variable 'ano' = 2008...")
df_master['ano'] = 2008

print("Exportando panel a Stata con etiquetas personalizadas...")

# Filtrar el diccionario de etiquetas solo para las variables que sobrevivieron al merge
etiquetas_finales = {col: etiquetas_stata.get(col, col) for col in df_master.columns}

pyreadstat.write_dta(
    df_master, 
    ruta_salida, 
    column_labels=etiquetas_finales
)

print("="*80)
print(f"¡ÉXITO! Base 2008 consolidada guardada en:\n{ruta_salida}")
print(f"Total de observaciones (Personas): {len(df_master)}")
print(f"Total de variables: {len(df_master.columns)}")
print("="*80)

Iniciando construcción de base 2008...

  [OK] caracteristicas_generales.dta cargado con 19 variables clave.
  [OK] hogares.dta cargado con 3 variables clave.
  [OK] ocupados.dta cargado con 18 variables clave.
  [OK] desocupados.dta cargado con 6 variables clave.
  [OK] otras_actividades.dta cargado con 4 variables clave.
  [OK] otros_ingresos.dta cargado con 12 variables clave.

Creando la variable 'ano' = 2008...
Exportando panel a Stata con etiquetas personalizadas...
¡ÉXITO! Base 2008 consolidada guardada en:
E:\ENTREGA TESIS\Doctorado\Data\Panel_2008_Preliminar.dta
Total de observaciones (Personas): 823814
Total de variables: 49


# creacion inicial de la base de datos completa v1

In [ ]:
import os
import pandas as pd
import pyreadstat

# =====================================================================
# 1. CONFIGURACIÓN DE RUTAS
# =====================================================================
ruta_base = r"E:\ENTREGA TESIS\Doctorado\Data"
ruta_salida_final = os.path.join(ruta_base, "Panel_GEIH_2008_2019.dta")

# =====================================================================
# 2. DICCIONARIO DE ETIQUETAS (BASADO EN TU CUADRO)
# =====================================================================
etiquetas_completas = {
    'ano': 'año', 'area': 'area', 'clase': 'clase', 'directorio': 'directorio', 
    'dpto': 'dpto', 'dsi': 'desocupados', 'esc': 'escolaridad', 'fex_c': 'factor', 
    'fex_c_2011': 'factor', 'ini': 'inactivos', 'mes': 'mes', 'oci': 'ocupados', 
    'orden': 'orden', 'p6020': 'sexo', 'p6040': 'edad', 'p6050': 'jefe_hogar', 
    'p6070': 'estado_civil', 'p6100': 'regimen_salud', 'p6110': 'quien_paga_regimen_salud', 
    'p6170': 'asistencia_escuela', 'p6210': 'nivel_edu', 'p6210s1': 'año_aprobado', 
    'p6220': 'diploma', 'secuencia_p': 'secuencia_p', 'oficio': 'oficio', 
    'rama2d_d': 'rama2d_d', 'rama4d_d': 'rama4d_d', 'ft': 'fuerza de trabajo', 
    'inglabo': 'buscar 200', 'p6430': 'posicion_ocupacional', 'p6450': 'tipo_contrato', 
    'p6585s4': 'subsidio_edu', 'p6765': 'formas_trabajo', 'p6772': 'registro_merca', 
    'p6772s1': 'año_renova', 'p6775': 'lleva_contabilidad', 'p6870': 'personas_empresa', 
    'p6920': 'pension', 'p6930': 'tipo_fondo', 'p6940': 'quien_paga_regimen_pension', 
    'rama2d': 'rama2d', 'rama2d_r4': 'rama2d_r4', 'rama4d': 'rama4d', 
    'p4030s1a1': 'estrato', 'p7480s8': 'cursos_cap', 'p7500s1a1': 'ing_arriendo', 
    'p7500s2a1': 'ing_pensiones', 'p7500s3a1': 'ing_aliment', 'p7510s1a1': 'ing_residentes', 
    'p7510s2a1': 'ing_noreside', 'p7510s3a1': 'ing_gob', 'p7510s5a1': 'ing_ahorr', 
    'p7510s6a1': 'ing_cesan', 'p7510s7a1': 'ing_otros'
}

# =====================================================================
# 3. LISTA DE VARIABLES POR MÓDULO (SEGÚN TU CUADRO)
# =====================================================================
config_vars = {
    "caracteristicas_generales": [
        'directorio', 'secuencia_p', 'orden', 'area', 'clase', 'dpto', 'dsi',
        'esc', 'fex_c', 'fex_c_2011', 'ini', 'mes', 'oci', 'p6020', 'p6040', 
        'p6050', 'p6070', 'p6100', 'p6110', 'p6170', 'p6210', 'p6210s1', 'p6220'
    ],
    "hogares": ['directorio', 'secuencia_p', 'p4030s1a1'],
    "ocupados": [
        'directorio', 'secuencia_p', 'orden', 'inglabo', 'oficio', 'p6430', 
        'p6450', 'p6585s4', 'p6765', 'p6772', 'p6772s1', 'p6775', 'p6870', 
        'p6920', 'p6930', 'p6940', 'rama2d', 'rama2d_r4', 'rama4d'
    ],
    "desocupados": ['directorio', 'secuencia_p', 'orden', 'dsi', 'rama2d_d', 'rama4d_d', 'fex_c'],
    "fuerza_trabajo": ['directorio', 'secuencia_p', 'orden', 'ft'],
    "otras_actividades": ['directorio', 'secuencia_p', 'orden', 'p7480s8'],
    "otros_ingresos": [
        'directorio', 'secuencia_p', 'orden', 'p7500s1a1', 'p7500s2a1', 'p7500s3a1', 
        'p7510s1a1', 'p7510s2a1', 'p7510s3a1', 'p7510s5a1', 'p7510s6a1', 'p7510s7a1'
    ]
}

# =====================================================================
# 4. PROCESO DE CARGA Y PEGUE POR AÑO
# =====================================================================
def procesar_ano(anho):
    folder_name = f"{anho}_"
    path_ano = os.path.join(ruta_base, folder_name)
    
    if not os.path.exists(path_ano):
        print(f"⚠️ Saltando {anho}: No se encontró la carpeta.")
        return None

    print(f"--- Procesando Año: {anho} ---")
    
    # A. Cargar Características Generales (Base Maestra)
    file_cg = os.path.join(path_ano, "caracteristicas_generales.dta")
    df_year, _ = pyreadstat.read_dta(file_cg)
    df_year.columns = [c.lower() for c in df_year.columns]
    
    # Limpieza de variable 'ano' original (evita ceros de 2008/2009)
    if 'ano' in df_year.columns:
        df_year = df_year.drop(columns=['ano'])
    
    # Filtrar solo variables que existan este año
    cols_cg = [c for c in config_vars["caracteristicas_generales"] if c in df_year.columns]
    df_year = df_year[cols_cg]
    
    # B. Pegar Hogares (Llaves: directorio, secuencia_p)
    file_hog = os.path.join(path_ano, "hogares.dta")
    if os.path.exists(file_hog):
        df_h, _ = pyreadstat.read_dta(file_hog)
        df_h.columns = [c.lower() for c in df_h.columns]
        cols_h = [c for c in config_vars["hogares"] if c in df_h.columns]
        df_year = pd.merge(df_year, df_h[cols_h], on=['directorio', 'secuencia_p'], how='left')

    # C. Pegar Módulos de Personas (Llaves: directorio, secuencia_p, orden)
    for modulo in ["ocupados", "desocupados", "fuerza_trabajo", "otras_actividades", "otros_ingresos"]:
        file_mod = os.path.join(path_ano, f"{modulo}.dta")
        if os.path.exists(file_mod):
            df_temp, _ = pyreadstat.read_dta(file_mod)
            df_temp.columns = [c.lower() for c in df_temp.columns]
            cols_temp = [c for c in config_vars[modulo] if c in df_temp.columns]
            
            # Merge por llaves de persona
            df_year = pd.merge(df_year, df_temp[cols_temp], 
                               on=['directorio', 'secuencia_p', 'orden'], 
                               how='left', suffixes=('', f'_{modulo}'))

    # D. Asignar el año correcto forzado
    df_year['ano'] = anho
    return df_year

# =====================================================================
# 5. EJECUCIÓN Y EXPORTACIÓN
# =====================================================================
lista_total = []

for a in range(2008, 2020):
    df_resultado = procesar_ano(a)
    if df_resultado is not None:
        lista_total.append(df_resultado)

# Concatenar todos los años
df_panel = pd.concat(lista_total, axis=0, ignore_index=True)

# Limpiar columnas duplicadas por sufijos (si existieran)
df_panel = df_panel.loc[:, ~df_panel.columns.duplicated()]

# Aplicar etiquetas de variables
etiquetas_finales = {col: etiquetas_completas.get(col, col) for col in df_panel.columns}

# Escribir archivo final
# Escribir archivo final (sin forzar la versión)
pyreadstat.write_dta(
    df_panel, 
    ruta_salida_final, 
    column_labels=etiquetas_finales
)

print("\n" + "="*60)
print(f"ÉXITO: Se ha generado el panel 2008-2019.")
print(f"Ruta: {ruta_salida_final}")
print(f"Total registros: {len(df_panel)}")
print("="*60)

--- Procesando Año: 2008 ---
--- Procesando Año: 2009 ---
--- Procesando Año: 2010 ---
--- Procesando Año: 2011 ---
--- Procesando Año: 2012 ---
--- Procesando Año: 2013 ---
--- Procesando Año: 2014 ---
--- Procesando Año: 2015 ---
--- Procesando Año: 2016 ---
--- Procesando Año: 2017 ---
--- Procesando Año: 2018 ---
--- Procesando Año: 2019 ---


Exception: Version not supported

para el codigo anterior 

In [2]:
# Escribir archivo final (sin forzar la versión)
pyreadstat.write_dta(
    df_panel, 
    ruta_salida_final, 
    column_labels=etiquetas_finales
)

Los datos de 2009 dan superiores a las bases, hay que mirar porque 

In [4]:
import os
import pandas as pd
import unicodedata
import zipfile
import tempfile
import pyreadstat

# =====================================================================
# 1. CONFIGURACIÓN
# =====================================================================
ruta_base = r"E:\ENTREGA TESIS\Doctorado\Data\2009"

# Diccionario de meses para mapear el número del ZIP al nombre exacto
meses_nombres = {
    '01': 'enero', '02': 'febrero', '03': 'marzo', '04': 'abril',
    '05': 'mayo', '06': 'junio', '07': 'julio', '08': 'agosto',
    '09': 'septiembre', '10': 'octubre', '11': 'noviembre', '12': 'diciembre'
}

def quitar_tildes(cadena):
    s = ''.join(c for c in unicodedata.normalize('NFD', str(cadena)) if unicodedata.category(c) != 'Mn')
    return s.lower()

def encontrar_archivo_en_zip(lista_archivos_zip, zona_busqueda, palabra_clave):
    clave_norm = quitar_tildes(palabra_clave)
    for archivo in lista_archivos_zip:
        archivo_norm = quitar_tildes(str(archivo).lower())
        if '.sav' in archivo_norm and zona_busqueda in archivo_norm and clave_norm in archivo_norm:
            return archivo
    return None

# =====================================================================
# 2. EXTRACCIÓN AL VUELO Y CONTEO
# =====================================================================
resultados = []

print("⏳ Contando observaciones directamente desde los ZIPs (tomará un par de minutos)...")

for mes_str, mes_texto in meses_nombres.items():
    ruta_zip = os.path.join(ruta_base, f"{mes_str}.zip")
    filas_cab = 0
    filas_res = 0
    
    if os.path.exists(ruta_zip):
        try:
            with zipfile.ZipFile(ruta_zip, 'r') as z:
                lista_archivos = z.namelist()
                
                # Identificamos los archivos exactos
                arch_cab = encontrar_archivo_en_zip(lista_archivos, 'cabecera', 'caract')
                arch_res = encontrar_archivo_en_zip(lista_archivos, 'resto', 'caract')
                
                # Función interna para extraer a tempfile, leer y contar
                def contar_filas(archivo_obj):
                    if not archivo_obj: return 0
                    
                    with tempfile.NamedTemporaryFile(delete=False, suffix=".sav") as tmp:
                        tmp.write(z.read(archivo_obj))
                        tmp_path = tmp.name
                        
                    try:
                        # Leemos el SAV temporal
                        df, _ = pyreadstat.read_sav(tmp_path)
                        return len(df)
                    except Exception as e:
                        print(f"      [Error leyendo {archivo_obj}]: {e}")
                        return 0
                    finally:
                        # Limpiamos el disco duro
                        if os.path.exists(tmp_path):
                            os.remove(tmp_path)
                            
                filas_cab = contar_filas(arch_cab)
                filas_res = contar_filas(arch_res)
                
                print(f"  [OK] {mes_texto.capitalize()} procesado.")
                
        except Exception as e:
            print(f"  [ERROR] Problema con ZIP {mes_str}: {e}")
            
    # Guardamos los resultados del mes
    resultados.append({'mes': mes_texto, 'cabecera': filas_cab, 'resto': filas_res})

# =====================================================================
# 3. RESULTADO FINAL (FORMATO TABLA)
# =====================================================================
df_resultado = pd.DataFrame(resultados).set_index('mes')

print("\n✅ TABLA FINAL DE OBSERVACIONES (PERSONAS) POR MES - 2009:")
print("=" * 45)
print(df_resultado)
print("=" * 45)

⏳ Contando observaciones directamente desde los ZIPs (tomará un par de minutos)...
  [OK] Enero procesado.
  [OK] Febrero procesado.
  [OK] Marzo procesado.
  [OK] Abril procesado.
  [OK] Mayo procesado.
  [OK] Junio procesado.
  [OK] Julio procesado.
  [OK] Agosto procesado.
  [OK] Septiembre procesado.
  [OK] Octubre procesado.
  [OK] Noviembre procesado.
  [OK] Diciembre procesado.

✅ TABLA FINAL DE OBSERVACIONES (PERSONAS) POR MES - 2009:
            cabecera  resto
mes                        
enero          58248   7209
febrero        60053   6730
marzo          60504   7046
abril          61271   7706
mayo           62610   6794
junio          59765   7396
julio          61609   7350
agosto         62843   6850
septiembre     62790   6510
octubre        61534   7096
noviembre      60695   7118
diciembre      59103   7412


In [5]:
#revision de observaciones de 2010
import os
import pandas as pd
import unicodedata
import zipfile
import tempfile
import pyreadstat

# =====================================================================
# 1. CONFIGURACIÓN DE RUTAS Y MESES
# =====================================================================
ruta_base = r"E:\ENTREGA TESIS\Doctorado\Data\2010\GEIH_Empalme_2010" 

# Mapeamos el nombre del ZIP al nombre del mes en texto para la tabla final
meses = {
    '1. Enero': 'enero', '2. Febrero': 'febrero', '3. Marzo': 'marzo', 
    '4. Abril': 'abril', '5. Mayo': 'mayo', '6. Junio': 'junio', 
    '7. Julio': 'julio', '8. Agosto': 'agosto', '9. Septiembre': 'septiembre', 
    '10. Octubre': 'octubre', '11. Noviembre': 'noviembre', '12. Diciembre': 'diciembre'
}

def quitar_tildes(cadena):
    try:
        s = ''.join(c for c in unicodedata.normalize('NFD', str(cadena)) if unicodedata.category(c) != 'Mn')
        return s.lower()
    except:
        return str(cadena).lower()

def encontrar_archivo_en_zip(lista_archivos_zip, lista_palabras_clave):
    for archivo in lista_archivos_zip:
        archivo_norm = quitar_tildes(archivo.lower())
        # Buscamos que sea .dta
        if '.dta' in archivo_norm:
            for palabra in lista_palabras_clave:
                if quitar_tildes(palabra) in archivo_norm:
                    return archivo
    return None

# =====================================================================
# 2. CONTEO DE OBSERVACIONES AL VUELO
# =====================================================================
resultados = []

print("⏳ Contando observaciones de Características Generales (Nacional) - 2010...")
print("-" * 60)

for mes_zip, mes_nombre in meses.items():
    ruta_zip = os.path.join(ruta_base, f"{mes_zip}.zip")
    filas = 0
    
    if os.path.exists(ruta_zip):
        try:
            with zipfile.ZipFile(ruta_zip, 'r') as z:
                # Buscamos el archivo de características generales
                archivo_objetivo = encontrar_archivo_en_zip(z.namelist(), ["caract"])
                
                if archivo_objetivo:
                    tmp_path = ""
                    try:
                        # Extraer a un temporal
                        with tempfile.NamedTemporaryFile(delete=False, suffix=".dta") as tmp:
                            tmp.write(z.read(archivo_objetivo))
                            tmp_path = tmp.name
                        
                        # Leer el .dta y contar
                        df, _ = pyreadstat.read_dta(tmp_path)
                        filas = len(df)
                        print(f"  [OK] {mes_nombre.capitalize()}: {filas:,} personas.")
                        
                    except Exception as e:
                        print(f"      [ERROR] leyendo {archivo_objetivo}: {e}")
                    finally:
                        # Limpieza
                        if tmp_path and os.path.exists(tmp_path):
                            os.remove(tmp_path)
                else:
                    print(f"  [AVISO] No se encontró el módulo 'caract' en el ZIP de {mes_nombre}.")
        except Exception as e:
            print(f"  [ERROR] abriendo el ZIP {mes_zip}: {e}")
    else:
        print(f"  [AVISO] No existe el archivo ZIP: {ruta_zip}")

    resultados.append({'mes': mes_nombre, 'total_personas': filas})

# =====================================================================
# 3. RESULTADO FINAL (TABLA)
# =====================================================================
df_resultado = pd.DataFrame(resultados).set_index('mes')

print("\n✅ TABLA FINAL DE OBSERVACIONES (PERSONAS) POR MES - 2010:")
print("=" * 45)
print(df_resultado)
print("=" * 45)
print(f"TOTAL AÑO 2010: {df_resultado['total_personas'].sum():,}")

⏳ Contando observaciones de Características Generales (Nacional) - 2010...
------------------------------------------------------------
  [OK] Enero: 68,654 personas.
  [OK] Febrero: 70,254 personas.
  [OK] Marzo: 69,809 personas.
  [OK] Abril: 71,148 personas.
  [OK] Mayo: 70,928 personas.
  [OK] Junio: 69,131 personas.
  [OK] Julio: 69,052 personas.
  [OK] Agosto: 71,424 personas.
  [OK] Septiembre: 70,146 personas.
  [OK] Octubre: 71,284 personas.
  [OK] Noviembre: 71,545 personas.
  [OK] Diciembre: 68,846 personas.

✅ TABLA FINAL DE OBSERVACIONES (PERSONAS) POR MES - 2010:
            total_personas
mes                       
enero                68654
febrero              70254
marzo                69809
abril                71148
mayo                 70928
junio                69131
julio                69052
agosto               71424
septiembre           70146
octubre              71284
noviembre            71545
diciembre            68846
TOTAL AÑO 2010: 842,221


# codigo pegado total v2

In [6]:
import os
import pandas as pd
import pyreadstat

# =====================================================================
# 1. CONFIGURACIÓN DE RUTAS
# =====================================================================
ruta_base = r"E:\ENTREGA TESIS\Doctorado\Data"
ruta_salida_final = os.path.join(ruta_base, "Panel_GEIH_2008_2019v2.dta")

# =====================================================================
# 2. DICCIONARIO DE ETIQUETAS (TU LISTA COMPLETA ORIGINAL)
# =====================================================================
etiquetas_completas = {
    'ano': 'año', 'area': 'area', 'clase': 'clase', 'directorio': 'directorio', 
    'dpto': 'dpto', 'dsi': 'desocupados', 'esc': 'escolaridad', 'fex_c': 'factor', 
    'fex_c_2011': 'factor', 'ini': 'inactivos', 'mes': 'mes', 'oci': 'ocupados', 
    'orden': 'orden', 'hogar': 'hogar', 'p6020': 'sexo', 'p6040': 'edad', 
    'p6050': 'jefe_hogar', 'p6070': 'estado_civil', 'p6100': 'regimen_salud', 
    'p6110': 'quien_paga_regimen_salud', 'p6170': 'asistencia_escuela', 
    'p6210': 'nivel_edu', 'p6210s1': 'año_aprobado', 'p6220': 'diploma', 
    'secuencia_p': 'secuencia_p', 'oficio': 'oficio', 'rama2d_d': 'rama2d_d', 
    'rama4d_d': 'rama4d_d', 'ft': 'fuerza de trabajo', 'inglabo': 'ingreso laboral', 
    'p6430': 'posicion_ocupacional', 'p6450': 'tipo_contrato', 'p6585s4': 'subsidio_edu', 
    'p6765': 'formas_trabajo', 'p6772': 'registro_merca', 'p6772s1': 'año_renova', 
    'p6775': 'lleva_contabilidad', 'p6870': 'personas_empresa', 'p6920': 'pension', 
    'p6930': 'tipo_fondo', 'p6940': 'quien_paga_regimen_pension', 'rama2d': 'rama2d', 
    'rama2d_r4': 'rama2d_r4', 'rama4d': 'rama4d', 'p4030s1a1': 'estrato', 
    'p7480s8': 'cursos_cap', 'p7500s1a1': 'ing_arriendo', 'p7500s2a1': 'ing_pensiones', 
    'p7500s3a1': 'ing_aliment', 'p7510s1a1': 'ing_residentes', 'p7510s2a1': 'ing_noreside', 
    'p7510s3a1': 'ing_gob', 'p7510s5a1': 'ing_ahorr', 'p7510s6a1': 'ing_cesan', 
    'p7510s7a1': 'ing_otros'
}

# =====================================================================
# 3. LISTA DE VARIABLES POR MÓDULO (TU LISTA COMPLETA ORIGINAL)
# =====================================================================
# NOTA: He agregado 'hogar' y 'mes' a todos los módulos para el pegue
config_vars = {
    "caracteristicas_generales": [
        'directorio', 'hogar', 'orden', 'mes', 'secuencia_p', 'area', 'clase', 
        'dpto', 'dsi', 'esc', 'fex_c', 'fex_c_2011', 'ini', 'oci', 'p6020', 
        'p6040', 'p6050', 'p6070', 'p6100', 'p6110', 'p6170', 'p6210', 'p6210s1', 'p6220'
    ],
    "hogares": ['directorio', 'hogar', 'mes', 'p4030s1a1'],
    "ocupados": [
        'directorio', 'hogar', 'orden', 'mes', 'inglabo', 'oficio', 'p6430', 
        'p6450', 'p6585s4', 'p6765', 'p6772', 'p6772s1', 'p6775', 'p6870', 
        'p6920', 'p6930', 'p6940', 'rama2d', 'rama2d_r4', 'rama4d'
    ],
    "desocupados": ['directorio', 'hogar', 'orden', 'mes', 'dsi', 'rama2d_d', 'rama4d_d', 'fex_c'],
    "fuerza_trabajo": ['directorio', 'hogar', 'orden', 'mes', 'ft'],
    "otras_actividades": ['directorio', 'hogar', 'orden', 'mes', 'p7480s8'],
    "otros_ingresos": [
        'directorio', 'hogar', 'orden', 'mes', 'p7500s1a1', 'p7500s2a1', 'p7500s3a1', 
        'p7510s1a1', 'p7510s2a1', 'p7510s3a1', 'p7510s5a1', 'p7510s6a1', 'p7510s7a1'
    ]
}

# =====================================================================
# 4. FUNCIÓN PROCESADORA (CON LLAVE MAESTRA HOGAR + MES)
# =====================================================================
def procesar_ano(anho):
    folder_name = f"{anho}_"
    path_ano = os.path.join(ruta_base, folder_name)
    
    if not os.path.exists(path_ano):
        return None

    print(f"--- Procesando Año: {anho} ---")
    
    # A. Cargar Características Generales
    file_cg = os.path.join(path_ano, "caracteristicas_generales.dta")
    df_year, _ = pyreadstat.read_dta(file_cg)
    df_year.columns = [c.lower() for c in df_year.columns]
    
    if 'ano' in df_year.columns: df_year = df_year.drop(columns=['ano'])
    
    cols_cg = [c for c in config_vars["caracteristicas_generales"] if c in df_year.columns]
    df_year = df_year[cols_cg]

    # DEFINICIÓN DE LLAVES BASADO EN TUS HALLAZGOS EN STATA
    llave_hogar = ['directorio', 'hogar', 'mes']
    llave_persona = ['directorio', 'hogar', 'orden', 'mes']

    # B. Pegar Hogares (1:M)
    file_hog = os.path.join(path_ano, "hogares.dta")
    if os.path.exists(file_hog):
        df_h, _ = pyreadstat.read_dta(file_hog)
        df_h.columns = [c.lower() for c in df_h.columns]
        df_h = df_h.drop_duplicates(subset=llave_hogar) # Limpieza fundamental
        cols_h = [c for c in config_vars["hogares"] if c in df_h.columns]
        df_year = pd.merge(df_year, df_h[cols_h], on=llave_hogar, how='left')

    # C. Pegar Módulos de Personas (1:1)
    for modulo in ["ocupados", "desocupados", "fuerza_trabajo", "otras_actividades", "otros_ingresos"]:
        file_mod = os.path.join(path_ano, f"{modulo}.dta")
        if os.path.exists(file_mod):
            df_temp, _ = pyreadstat.read_dta(file_mod)
            df_temp.columns = [c.lower() for c in df_temp.columns]
            df_temp = df_temp.drop_duplicates(subset=llave_persona) # Limpieza para que no aumenten filas
            cols_temp = [c for c in config_vars[modulo] if c in df_temp.columns]
            df_year = pd.merge(df_year, df_temp[cols_temp], on=llave_persona, how='left')

    df_year['ano'] = anho
    return df_year

# =====================================================================
# 5. EJECUCIÓN FINAL
# =====================================================================
lista_total = []
for a in range(2008, 2020):
    df_resultado = procesar_ano(a)
    if df_resultado is not None:
        lista_total.append(df_resultado)

df_panel = pd.concat(lista_total, axis=0, ignore_index=True)

# Eliminar duplicados de columnas (por si acaso) y aplicar etiquetas
df_panel = df_panel.loc[:, ~df_panel.columns.duplicated()]
etiquetas_finales = {col: etiquetas_completas.get(col, col) for col in df_panel.columns}

# Escribir archivo final con etiquetas
pyreadstat.write_dta(df_panel, ruta_salida_final, column_labels=etiquetas_finales)

print(f"\n✅ ¡PROCESO TERMINADO! Se incluyeron todas tus variables originales.")
print(f"Total registros en el panel: {len(df_panel):,}")

--- Procesando Año: 2008 ---
--- Procesando Año: 2009 ---
--- Procesando Año: 2010 ---
--- Procesando Año: 2011 ---
--- Procesando Año: 2012 ---
--- Procesando Año: 2013 ---
--- Procesando Año: 2014 ---
--- Procesando Año: 2015 ---
--- Procesando Año: 2016 ---
--- Procesando Año: 2017 ---
--- Procesando Año: 2018 ---
--- Procesando Año: 2019 ---

✅ ¡PROCESO TERMINADO! Se incluyeron todas tus variables originales.
Total registros en el panel: 9,901,637
